# **0**. Path configuration

Intermediate files are stored under `Intermediate Files/` on Google Drive.
**Run the next cell before running any other cell.**

| Folder | Purpose |
|--------|------|
| `Raw_Data/` | Raw data |
| `Core_Process/` | Main pipeline for the benchmark sample |
| `Undetermined_Process/` | Pipeline for undetermined partnerships |
| `Humanly_Picked/` | Hand-labelled samples |
| `Models/` | Trained models and embeddings (`.npy`) |

In [ ]:
# ===== Global path configuration =====
HOME_DIR = "/content/drive/MyDrive/"
INTERMEDIATE_DIR = HOME_DIR + "NEW_Intermediate Files/"

RAW_DIR = INTERMEDIATE_DIR + "Raw_Data/"
CORE_DIR = INTERMEDIATE_DIR + "Core_Process/"
UNDETERMINED_DIR = INTERMEDIATE_DIR + "Undetermined_Process/"
HUMANLY_PICKED_DIR = INTERMEDIATE_DIR + "Humanly_Picked_Data/"
MODELS_DIR = INTERMEDIATE_DIR + "Models/"
RESULTS_DIR = INTERMEDIATE_DIR + "Results/"

SAVED_MODEL_NAME = "BERT_clean_suffix_model"
EMB_CACHE_FILE = "fund_embeddings_bge_m3.npy"

print("RAW_DIR:", RAW_DIR)
print("CORE_DIR:", CORE_DIR)
print("UNDETERMINED_DIR:", UNDETERMINED_DIR)
print("HUMANLY_PICKED_DIR:", HUMANLY_PICKED_DIR)
print("MODELS_DIR:", MODELS_DIR)


from google.colab import drive
drive.mount('/content/drive')
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
import pandas as pd
import numpy as np
import torch
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import difflib

# **1.** Benchmark: prefix cleaning
*Remove boilerplate prefixes such as “一般许可：” (general permit:) and “一般经营项目是：” (general business scope:)*





In [ ]:
INPUT_DIR = RAW_DIR
OUTPUT_DIR = CORE_DIR
INPUT_FILE = 'benchmark.dta'
OUTPUT_FILE = 'benchmark_cleaned.csv'

import re
import pandas as pd

# ===== Step 1: replicate decompose_general_permit.do =====

GENERAL = ["一般经营项目是：","一般经营项目是:","一般经营项目：","一般经营项目:",
           "一般经营项目 ：","一般经营项目 :","一般经营项目（","一般经营项目(",
           "一般项目：","一般项目:","一般项目 ：","一般项目 :",
           "一般事项：","一般事项:",
           "一般经营范围：","一般经营范围:","一般经营范围 ：","一般经营范围 :"]
PERMIT  = ["许可经营项目是：","许可经营项目是:","许可经营项目：","许可经营项目:",
           "许可经营项目 ：","许可经营项目 :","许可经营项目（","许可经营项目(",
           "许可项目：","许可项目:","许可项目 ：","许可项目 :",
           "许可经营范围：","许可经营范围:","许可经营范围 ：","许可经营范围 :"]
DISPLAY = ["自主展示（特色）项目：","自主展示（特色）项目:",
           "自主展示(特色)项目：","自主展示(特色)项目:",
           "自主展示项目：","自主展示项目:",
           "合伙协议记载的经营范围：","合伙协议记载的经营范围:",
           "协议记载的经营范围：","协议记载的经营范围:"]
def first_pos(s, variants):
    for v in variants:
        p = s.find(v)
        if p >= 0:
            return p
    return -1
def decompose(scope):
    """Split scope into [(j, segment)], j: 1=general 2=permit 3=other 4=display.
    Labels stay at the head of their segment (stripped later by clean step)."""
    if pd.isna(scope):
        return []
    s = str(scope)
    hits = []
    for j, vs in [(1, GENERAL), (2, PERMIT), (4, DISPLAY)]:
        p = first_pos(s, vs)
        if p >= 0:
            hits.append((p, j))
    if not hits:
        return [(3, s.strip())]
    hits.sort()
    segs = []
    if hits[0][0] > 0:
        segs.append((3, s[:hits[0][0]].strip()))
    for k, (pos, j) in enumerate(hits):
        end = hits[k+1][0] if k+1 < len(hits) else len(s)
        segs.append((j, s[pos:end].strip()))
    return [(j, t) for j, t in segs if t != '']

# ===== Step 2: replicate clean_label_prefix.do =====

_c1 = re.compile(r'^(一般|许可)(经营)?(项目|事项|范围)(是)?\s*[：:]\s*')
_c2 = re.compile(r'^(一般|许可)(经营)?(项目|事项|范围)(是)?(?=[（(])')
_c3 = re.compile(r'^(合伙)?(协议记载的)?(经营范围|主营业务|主要经营|经营项目|服务)\s*[：:]\s*')
_c3b = re.compile(r'^合伙协议记载的经营范围[:：]\s*')
_c3c = re.compile(r'^章程记载的经营范围[:：]\s*')
_c4 = re.compile(r'^[^：:]{1,18}项目\s*[：:]\s*')

def clean_label_prefix(text):
    s = str(text)
    for pat in (_c1, _c2, _c3, _c3b, _c3c, _c4):
        s = pat.sub('', s, count=1)
    return s.strip()   # Unicode-aware, incl. \u3000 — equivalent to ustrtrim

# ===== Pipeline =====
df = pd.read_stata(INPUT_DIR + INPUT_FILE)
rows = []
for _, r in df.iterrows():
    for j, seg in decompose(r['scope']):
        rows.append({'id': r['id'], 'j': j, 'biz_main': seg,
                     '原文件导入名称': r['原文件导入名称'],
                     'indicator': r['indicator'],
                     '统一社会信用代码': r['统一社会信用代码']})
long = pd.DataFrame(rows).sort_values(['id', 'j'])
long['biz_main_clean'] = long['biz_main'].apply(clean_label_prefix)
long.to_csv(OUTPUT_DIR + OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f'Input file is: {INPUT_FILE}')
print(f'Output file is: {OUTPUT_FILE}')

# **2.** Train and save BERT

*Requires a Google Colab L4 GPU*

In [ ]:
LABEL_FILE = CORE_DIR + "benchmark_cleaned.csv"
LABELED_FILE = HUMANLY_PICKED_DIR + "disclaimer_random sample.xlsx"
label_data = pd.read_csv(LABEL_FILE)

## Read the labelled data

The labelled data are in `LABELED_FILE`, currently

In [ ]:
# Label the training set
label_data_raw = pd.read_excel(LABELED_FILE)
label_data_raw = label_data_raw

def build_char_labels(row):
    bizdesc = str(row['bizdesc'])
    bp = str(row['bizdesc_bp']).strip()

    # empty bp -> the whole text is suffix
    if not bp or bp == 'nan':
        return ['S'] * len(bizdesc)

    # use SequenceMatcher to find the part of bizdesc that was deleted
    matcher = difflib.SequenceMatcher(None, bizdesc, bp)
    labels = ['S'] * len(bizdesc)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            for k in range(i1, i2):
                labels[k] = 'O'

    return labels

label_data_raw['char_labels'] = label_data_raw.apply(build_char_labels, axis=1)
label_data = label_data_raw[label_data_raw['bizdesc'].notna()].copy()
label_data = label_data.rename(columns={'bizdesc': 'scope'})

# Check
row = label_data.iloc[1]
print(f"Text: {str(row['scope'])[:80]}...")
print(f"Labels: {''.join(row['char_labels'][:80])}...")
print(f"\nLabelled rows: {len(label_data)}")

## Train and save the BERT model for later use

In [ ]:
!pip install transformers datasets -q  # transformers must be installed separately in Google Colab

MODEL_NAME = "BERT_clean_suffix_model"

from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

# ===== Label definitions =====
label_list = ["O", "S"]  # O = main content, S = suffix
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

# ===== Load the model =====
MODEL_NAME = "hfl/chinese-roberta-wwm-ext"  # base version
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
)
# ===== Train =====
def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["scope"],
        max_length=512,
        truncation=True,
        is_split_into_words=False,
    )
    all_labels = []
    for i, char_label_list in enumerate(examples["char_labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        labels = []
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid < len(char_label_list):
                labels.append(label2id[char_label_list[wid]])
            else:
                labels.append(-100)
        all_labels.append(labels)
    tokenized["labels"] = all_labels
    return tokenized

train_split, test_split = train_test_split(label_data, test_size=0.1, random_state=42)

# ===== Augment the training set only (no leakage into the test set) =====
augmented = []
for _, row in train_split.iterrows():
    text = str(row['scope'])
    labels = row['char_labels']
    s_count = 0
    in_s = False
    for lb in labels:
        if lb == 'S' and not in_s:
            s_count += 1
            in_s = True
        elif lb != 'S':
            in_s = False
    if s_count >= 2:
        for _ in range(3):
            augmented.append({'scope': text, 'char_labels': labels})

aug_df = pd.DataFrame(augmented)
train_split = pd.concat([train_split, aug_df], ignore_index=True)
print(f"Training set after augmentation: {len(train_split)}, test set: {len(test_split)} (not augmented)")

train_dataset = Dataset.from_pandas(train_split[['scope', 'char_labels']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_split[['scope', 'char_labels']].reset_index(drop=True))

train_dataset = train_dataset.map(tokenize_and_align, batched=True, remove_columns=["scope", "char_labels"])
test_dataset = test_dataset.map(tokenize_and_align, batched=True, remove_columns=["scope", "char_labels"])

print(f"Training set: {len(train_dataset)}, test set: {len(test_dataset)}")

from torch import nn
import torch.nn.functional as F

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = F.cross_entropy(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
            ignore_index=-100,
            reduction='none'
        )
        pt = torch.exp(-loss)
        focal_loss = ((1 - pt) ** 2 * loss).mean()
        return (focal_loss, outputs) if return_outputs else focal_loss

trainer = FocalLossTrainer(
    model=model,
    args=TrainingArguments(
        output_dir="/content/model_suffix",
        warmup_ratio=0.1,  #warmup
        num_train_epochs=12,
        per_device_train_batch_size=16,
        learning_rate=1e-5,
        bf16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=20,
        report_to="none",
    ),
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)

trainer.train()

model.save_pretrained(MODELS_DIR + SAVED_MODEL_NAME)
tokenizer.save_pretrained(MODELS_DIR + SAVED_MODEL_NAME)
print(f'Model saved to {MODELS_DIR}{SAVED_MODEL_NAME}')

# Benchmark: suffix cleaning (BERT)

1. Predict the suffix (disclaimer) with the model: sliding-window inference, and fill in segments that are too short; for example, the two O's in "...SSOOSSSS..." are most likely misclassified
2. Correct BERT's imprecise boundaries with suffix post-processing

## Load and test the model

In [ ]:
!pip install transformers datasets -q
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

MODEL_PATH = MODELS_DIR + "cleaned_suffix_model最终清洗suffix的模型"

label_list = ["O", "S"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

# ===== Load the saved model =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded: {MODEL_PATH}, device: {device}")

# ===== Load the labelled data =====
label_data_raw = pd.read_excel(LABELED_FILE)

def build_char_labels(row):
    bizdesc = str(row['bizdesc'])
    bp = str(row['bizdesc_bp']).strip()
    if not bp or bp == 'nan':
        return ['S'] * len(bizdesc)
    matcher = difflib.SequenceMatcher(None, bizdesc, bp)
    labels = ['S'] * len(bizdesc)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            for k in range(i1, i2):
                labels[k] = 'O'
    return labels

# ===== Sliding-window inference =====
def predict_suffix(text, max_len=510, stride=128):
    if len(text) <= max_len:
        enc = tokenizer(text, return_tensors="pt", max_length=512, truncation=True,
                        return_offsets_mapping=True)
        offsets = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()
        char_labels = ['O'] * len(text)
        for tok_idx, (start, end) in enumerate(offsets):
            if start == end:
                continue
            label = id2label[preds[tok_idx]]
            for ci in range(start, end):
                char_labels[ci] = label
    else:
        char_labels = ['O'] * len(text)
        char_scores = [0.0] * len(text)
        starts = list(range(0, len(text) - max_len + 1, stride))
        if starts[-1] + max_len < len(text):
            starts.append(len(text) - max_len)
        for s in starts:
            chunk = text[s:s + max_len]
            enc = tokenizer(chunk, return_tensors="pt", max_length=512, truncation=True,
                            return_offsets_mapping=True)
            offsets = enc.pop("offset_mapping")[0].tolist()
            enc = {k: v.to(model.device) for k, v in enc.items()}
            with torch.no_grad():
                logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[0].cpu()
            preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()
            for tok_idx, (cs, ce) in enumerate(offsets):
                if cs == ce:
                    continue
                confidence = probs[tok_idx].max().item()
                gpos = s + cs
                if gpos < len(text) and confidence > char_scores[gpos]:
                    label = id2label[preds[tok_idx]]
                    for ci in range(gpos, min(s + ce, len(text))):
                        char_labels[ci] = label
                        char_scores[ci] = confidence

    max_gap = 5
    i = 0
    while i < len(char_labels):
        if char_labels[i] == 'S':
            j = i
            while j < len(char_labels) and char_labels[j] == 'S':
                j += 1
            gap_start = j
            k = j
            while k < len(char_labels) and char_labels[k] == 'O':
                k += 1
            gap_len = k - gap_start
            if k < len(char_labels) and char_labels[k] == 'S' and gap_len <= max_gap:
                for p in range(gap_start, k):
                    char_labels[p] = 'S'
            i = j
        else:
            i += 1

    suffix_parts, main_parts = [], []
    current_label, current_text = None, []
    for ch, lb in zip(text, char_labels):
        if lb != current_label:
            if current_label == 'S':
                suffix_parts.append(''.join(current_text))
            current_label = lb
            current_text = [ch]
        else:
            current_text.append(ch)
    if current_label == 'S':
        suffix_parts.append(''.join(current_text))

    current_label, current_text = None, []
    for ch, lb in zip(text, char_labels):
        if lb != current_label:
            if current_label == 'O':
                main_parts.append(''.join(current_text))
            current_label = lb
            current_text = [ch]
        else:
            current_text.append(ch)
    if current_label == 'O':
        main_parts.append(''.join(current_text))

    return {'suffix': ''.join(suffix_parts), 'main': ''.join(main_parts)}

def strip_punct(s):
    return re.sub(r'[。.；;，,、：:！!？?（()）\[\]【】{}「」""''《》\s\^*#〓\n\r]', '', s)

def postprocess_suffix(pred_suffix, original_text):
    open_brackets = '（([【'
    close_brackets = '）)]】'
    close_to_open = {'）':'（', ')':'(', ']':'[', '】':'【'}
    keywords = ['法', '规', '律', '中国', '国家', '须', '批准', '限制', '禁止', '除外',
                '不得', '不含', '除', '凭', '经营活动',
                '无需', '审批', '许可', '市场主体', '备案']

    stack = []
    unmatched_close = []
    for idx, ch in enumerate(pred_suffix):
        if ch in open_brackets:
            stack.append(idx)
        elif ch in close_brackets:
            if stack:
                stack.pop()
            else:
                unmatched_close.append(idx)

    replacements = []
    for idx in unmatched_close:
        close_ch = pred_suffix[idx]
        open_ch = close_to_open.get(close_ch, '（')
        prev_end = 0
        for j in range(idx - 1, -1, -1):
            if pred_suffix[j] in close_brackets:
                prev_end = j + 1
                break
        for length in range(min(8, idx), 1, -1):
            snippet = pred_suffix[idx-length:idx]
            search = snippet + close_ch
            pos = original_text.find(search)
            if pos >= 0:
                actual_close = pos + len(search) - 1
                d = 1
                j = actual_close - 1
                while j >= 0 and d > 0:
                    if original_text[j] == close_ch: d += 1
                    elif original_text[j] == open_ch: d -= 1
                    j -= 1
                if d == 0:
                    full_bracket = original_text[j+1:actual_close+1]
                    replacements.append((prev_end, idx+1, full_bracket))
                break

    fixed = pred_suffix
    for start, end, replacement in sorted(replacements, key=lambda x: x[0], reverse=True):
        fixed = fixed[:start] + replacement + fixed[end:]

    output = []
    i = 0
    current = []
    in_bracket = 0
    while i < len(fixed):
        if fixed[i] in open_brackets:
            if current and in_bracket == 0:
                chunk = ''.join(current)
                if any(k in chunk for k in keywords):
                    output.append(chunk)
                current = []
            in_bracket += 1
            current.append(fixed[i])
        elif fixed[i] in close_brackets:
            current.append(fixed[i])
            in_bracket -= 1
            if in_bracket <= 0:
                in_bracket = 0
                output.append(''.join(current))
                current = []
        else:
            current.append(fixed[i])
        i += 1
    if current:
        chunk = ''.join(current)
        if in_bracket > 0 or any(k in chunk for k in keywords):
            output.append(chunk)
    return ''.join(output)

# ===== Run the test =====
_, test_clean = train_test_split(label_data_raw, test_size=0.1, random_state=42)
test_clean = test_clean.copy()
test_clean['char_labels'] = test_clean.apply(build_char_labels, axis=1)
test_clean = test_clean.rename(columns={'bizdesc': 'scope'})

exact_match = 0
content_match = 0
content_errors = []
total = len(test_clean)

for i, (_, row) in enumerate(test_clean.iterrows()):
    text = str(row['scope'])
    pred = predict_suffix(text)
    pred_suffix = pred['suffix']

    true_parts = []
    cur_label = None
    cur_text = []
    for ch, lb in zip(text, row['char_labels']):
        if lb != cur_label:
            if cur_label == 'S':
                true_parts.append(''.join(cur_text))
            cur_label = lb
            cur_text = [ch]
        else:
            cur_text.append(ch)
    if cur_label == 'S':
        true_parts.append(''.join(cur_text))
    true_suffix = ''.join(true_parts)

    if pred_suffix == true_suffix:
        exact_match += 1
        content_match += 1
    elif postprocess_suffix(pred_suffix, text) == postprocess_suffix(true_suffix, text):
        content_match += 1
        print(f"\n{'='*60}")
        print(f"✅ Row {i} (match after completion)")
        print(f"  S predicted (raw): {pred_suffix}")
        print(f"  S predicted (completed): {postprocess_suffix(pred_suffix, text)}")
        print(f"  S true (raw): {true_suffix}")
        print(f"  S true (completed): {postprocess_suffix(true_suffix, text)}")
    else:
        content_errors.append({
            'row': i,
            'text': text,
            'pred_raw': pred_suffix,
            'true_raw': true_suffix,
            'pred_post': postprocess_suffix(pred_suffix, text),
            'true_post': postprocess_suffix(true_suffix, text),
        })

print(f"\n{'='*60}")
print(f"Exact match: {exact_match}/{total} ({exact_match/total*100:.1f}%)")
print(f"Content match (ignoring punctuation): {content_match}/{total} ({content_match/total*100:.1f}%)")
print(f"Content errors: {len(content_errors)}")

for e in content_errors:
    print(f"\n{'='*60}")
    print(f"❌ Row {e['row']}")
    print(f"  Text: {e['text']}")
    print(f"  S predicted (raw): {e['pred_raw']}")
    print(f"  S predicted (completed): {e['pred_post']}")
    print(f"  S true (raw): {e['true_raw']}")
    print(f"  S true (completed): {e['true_post']}")

## Run BERT on the full benchmark data

1. Clean with the trained BERT
2. Post-cleaning, mainly to fix brackets

In [ ]:
INPUT_DIR = CORE_DIR
OUTPUT_DIR = CORE_DIR
INPUT_FILE = "benchmark_cleaned.csv"
OUTPUT_FILE = "test_bert_benchmark_cleaned.xlsx"

LABEL_FILE = INPUT_DIR + INPUT_FILE
label_data = pd.read_csv(LABEL_FILE)
all_texts = label_data['biz_main_clean'].astype(str).tolist()
predictions = [predict_suffix(t) for t in tqdm(all_texts)]

label_data['pred_main'] = [p['main'] for p in predictions]
label_data['pred_suffix'] = [p['suffix'] for p in predictions]
label_data['pred_suffix_post'] = [postprocess_suffix(p['suffix'], t) for p, t in zip(predictions, all_texts)]
label_data.to_excel(OUTPUT_DIR + OUTPUT_FILE, index=False)
print(f"Done: {len(label_data)} rows")

print(f'Input file is: {INPUT_FILE}')
print(f'Output file is: {OUTPUT_FILE}')

## Define and run the post-processing and fixing steps

In [ ]:
INPUT_DIR = CORE_DIR
OUTPUT_DIR = CORE_DIR
INPUT_FILE = "test_bert_benchmark_cleaned.xlsx"
OUTPUT_FILE = "test_Bert_Benchmark_Post_fix.xlsx"

import re
import pandas as pd
from difflib import SequenceMatcher

# ===== New fixing process: a BERT-faithful layer plus a regex fallback layer (with row-level flags) =====
OPENS  = set('(（[【')
CLOSES = set(')）]】')
NO_CONTENT = re.compile(r'^[。；;，,、\s.．()（）\[\]【】]*$')
DISCLAIMER_KW = re.compile(
    r'依法须经批准|凭营业执照|登记备案后方可|批准后方可开展经营|批准后依批准的内容|'
    r'除许可业务外|市场主体自主选择经营|国务院决定规定|不得以任何方式公开募集|'
    r'无需行政审批|外商投资准入特别管理措施|应取得相关部门许可后方可经营')
INNER_BLOCK = re.compile(r'[（(【\[][^（()）【】\[\]]*[)）\]】]')
BOUNDARY = set('。；;，,、.．：:！!？?＂"“” \t^〓*＊#＃=＝※') | OPENS | CLOSES

BRACKET_NUM = re.compile(r'[(（]\s*(?:[0-9]{1,2}|[一二三四五六七八九十]{1,3})\s*[)）]')
SEG_NUM = re.compile(r'(?:^|(?<=[。；;，,、]))\s*(?:[0-9]{1,2}|[一二三四五六七八九十]{1,3})\s*(?:[、::：]|[..](?![0-9]))')
SEG_LABEL = re.compile(r'(?:^|(?<=[。；;，,]))(?:许可经营项目|一般经营项目|许可项目|一般项目|经营范围)(?:是)?[:：]?'
                       r'|(?:^|(?<=[。；;，,]))(?:服务|资本市场服务)(?:从)?[:：]')
DATE = r'\d{4}年\d{1,2}月\d{1,2}日'
DURATION = re.compile(
    rf'(?:合伙|营业|经营|存续)(?:企业)?期限[:：]?(?:为|是)?'
    rf'(?:自(?:{DATE}|设立之日)起?)?'
    rf'(?:至{DATE}|[一二三四五六七八九十百\d]+年|长期)?止?')

def clean_bcd(s):
    s = BRACKET_NUM.sub('；', s)
    s = SEG_LABEL.sub('', s)
    s = SEG_NUM.sub('', s)
    s = SEG_LABEL.sub('', s)     # numbers and labels can hide each other: strip each once, then one more round
    s = _drop_orphan_brackets(s)
    s = DURATION.sub('', s)
    return s

def split_suffix_pieces(s):
    pieces, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch in OPENS:
            if depth == 0 and i > start:
                pieces.append(s[start:i]); start = i
            depth += 1
        elif ch in CLOSES:
            if depth > 0:
                depth -= 1
                if depth == 0:
                    pieces.append(s[start:i+1]); start = i + 1
    if start < len(s):
        pieces.append(s[start:])
    return pieces

def _balanced(s):
    st = []
    for ch in s:
        if ch in OPENS: st.append(ch)
        elif ch in CLOSES:
            if not st: return False
            st.pop()
    return not st

def _orphan_idx(s):
    bad, st = set(), []
    for i, ch in enumerate(s):
        if ch in OPENS: st.append(i)
        elif ch in CLOSES:
            if st: st.pop()
            else: bad.add(i)
    bad.update(st)
    return bad

def _drop_orphan_brackets(s):
    bad = _orphan_idx(s)
    return ''.join(ch for i, ch in enumerate(s) if i not in bad)

def sweep_disclaimers_count(text):
    """Regex sweep as the last layer, with a count: returns (cleaned text, number of boilerplate blocks removed)"""
    n = 0
    prev = None
    while prev != text:
        prev = text
        for m in INNER_BLOCK.finditer(text):
            if DISCLAIMER_KW.search(m.group()):
                text = text[:m.start()] + text[m.end():]
                n += 1
                break
    return text, n

def tidy_punct(s):
    """Tidy the punctuation left after deletions"""
    s = re.sub(r'[\^〓＊*＃#＝=※]+', '；', s)              # garbled separators -> ；
    s = re.sub(r'([（(【\[])\s*[。；;，,、.]+', r'\1', s)    # punctuation right inside an opening bracket
    s = re.sub(r'[。；;，,、.]+\s*([)）\]】])', r'\1', s)    # punctuation right inside a closing bracket
    def _collapse(m):                                       # collapse repeated punctuation, keeping the strongest mark
        run = m.group()
        if '。' in run or '.' in run: return '。'
        if '；' in run or ';' in run: return '；'
        if '，' in run or ',' in run: return '，'
        if '、' in run: return '、'
        return ''                                           # whitespace only
    s = re.sub(r'[。；;，,、.\s]{2,}', _collapse, s)
    s = s.lstrip('。；;，,、. ')                             # strip leading punctuation
    if s and s[-1] in '；;，,、.':
        s = s[:-1] + '。'                                   # end with a full stop
    s = re.sub(r'（）|\(\)|【】|\[\]', '', s)                # empty bracket pairs
    return s.strip()

def remove_suffixes_v6_flagged(text, suffix_str, min_span=3, min_cover=0.8):
    """v6 with row-level flags. Returns a Series:
       post      cleaned text
       n_exact   blocks deleted exactly as a whole (layer 1, BERT labels applied as they are)
       n_align   phantom blocks handled by sequence alignment (layer 2)
       n_giveup  phantom blocks skipped for low alignment coverage (better to under-delete)
       n_sweep   boilerplate blocks removed by the regex sweep (layer 3)"""
    if pd.isna(text):
        return pd.Series([text, 0, 0, 0, 0],
                         index=['post', 'n_exact', 'n_align', 'n_giveup', 'n_sweep'])
    r = str(text)
    n_exact = n_align = n_giveup = 0
    if not (pd.isna(suffix_str) or not str(suffix_str).strip()):
        failed = []
        for p in split_suffix_pieces(str(suffix_str)):
            if NO_CONTENT.match(p):
                continue
            c = p.strip('。；;，,、 ')
            if not _balanced(p) or (not any(ch in OPENS | CLOSES for ch in p)  # added 7/14 to avoid deleting suffix segments that are too long
                                    and len(c) < 6):        #
                failed.append(p); continue                        #
            if p in r:
                r = r.replace(p, '', 1); n_exact += 1
            elif c in r and len(c) >= 4:
                r = r.replace(c, '', 1); n_exact += 1
            else:
                failed.append(p)
        for p in failed:
            sm = SequenceMatcher(None, p, r, autojunk=False)
            blocks = [b for b in sm.get_matching_blocks() if b.size >= min_span]
            if sum(b.size for b in blocks) < min_cover * len(p):
                n_giveup += 1
                continue
            n_align += 1
            for b in sorted(blocks, key=lambda b: b.b, reverse=True):
                s0, s1 = b.b, b.b + b.size
                span = r[s0:s1]
                bad = _orphan_idx(span)
                while bad and (0 in bad or len(span) - 1 in bad):
                    if 0 in bad: s0 += 1
                    else: s1 -= 1
                    span = r[s0:s1]; bad = _orphan_idx(span)
                if len(span) < min_span or not _balanced(span):
                    continue
                if len(span) < 10:      # short span: both ends must fall on natural boundaries, so no characters are cut from inside a word
                    left_ok  = s0 == 0 or r[s0-1] in BOUNDARY or span[0] in OPENS
                    right_ok = s1 >= len(r) or r[s1] in BOUNDARY or span[-1] in CLOSES
                    if not (left_ok and right_ok):
                        continue
                r = r[:s0] + r[s1:]
        if failed:
            r = _drop_orphan_brackets(r)
    r, n_sweep = sweep_disclaimers_count(r)
    return pd.Series([tidy_punct(clean_bcd(r)), n_exact, n_align, n_giveup, n_sweep],
                     index=['post', 'n_exact', 'n_align', 'n_giveup', 'n_sweep'])

# ===== Pipeline =====
df = pd.read_excel(INPUT_DIR + INPUT_FILE)
print(f"Read {len(df)} rows")

old_post = df['post_biz_main_clean'].fillna('').astype(str) \
           if 'post_biz_main_clean' in df.columns else None

res = df.apply(lambda r: remove_suffixes_v6_flagged(
    r['biz_main_clean'], r['pred_suffix_post']), axis=1)
df['post_biz_main_clean'] = res['post']
df['fix_n_exact']  = res['n_exact'].astype(int)
df['fix_n_align']  = res['n_align'].astype(int)     # >0 = a phantom block in this row was realigned
df['fix_n_giveup'] = res['n_giveup'].astype(int)    # >0 = a suffix block in this row was skipped (may remain)
df['fix_n_sweep']  = res['n_sweep'].astype(int)     # >0 = the regex fallback was used

# ===== Change statistics =====
print("=" * 60)
print(f"Layer 1, exact block deletion: {(df['fix_n_exact'] > 0).sum()} rows, {df['fix_n_exact'].sum()} blocks")
print(f"Layer 2, phantom blocks aligned: {(df['fix_n_align'] > 0).sum()} rows, {df['fix_n_align'].sum()} blocks")
print(f"         phantom blocks skipped: {(df['fix_n_giveup'] > 0).sum()} rows, {df['fix_n_giveup'].sum()} blocks (suffix may remain)")
print(f"Layer 3, regex sweep of boilerplate: {(df['fix_n_sweep'] > 0).sum()} rows, {df['fix_n_sweep'].sum()} blocks")

new_post = df['post_biz_main_clean'].fillna('').astype(str)
if old_post is not None:
    changed = (old_post != new_post)
    print(f"Rows that differ from the old post_biz_main_clean: {changed.sum()} / {len(df)}")
    for _, row in df[changed].head(10).iterrows():
        print("=" * 60)
        print(f"  suffix:  {row['pred_suffix_post']}")
        print(f"  old clean: {old_post[row.name]}")
        print(f"  new clean: {row['post_biz_main_clean']}")

# ===== Self-check: brackets and boilerplate =====
def _bal_bad(s):
    st, uc = [], 0
    for ch in str(s):
        if ch in OPENS: st.append(ch)
        elif ch in CLOSES:
            if st: st.pop()
            else: uc += 1
    return len(st) + uc > 0

print("=" * 60)
print(f"Rows with unbalanced brackets: {new_post.apply(_bal_bad).sum()} (already broken in the raw text: "
      f"{df['biz_main_clean'].fillna('').astype(str).apply(_bal_bad).sum()})")
print(f"Rows still containing '依法须经批准' (subject to approval by law): {new_post.str.contains('依法须经批准').sum()}")

# ===== Deletion audit: warn when cleaning removed text that BERT did not label =====
raw = df['biz_main_clean'].fillna('').astype(str)
sfx = df['pred_suffix_post'].fillna('').astype(str)
for kw in ['股权投资', '创业投资', '投资管理', '投资咨询', '资产管理', '基金管理']:
    lost_not_bert = (raw.str.contains(kw, regex=False)
                     & ~new_post.str.contains(kw, regex=False)
                     & ~sfx.str.contains(kw, regex=False))
    status = "OK" if lost_not_bert.sum() == 0 else "!!!"
    print(f"Audit [{status}] '{kw}': {lost_not_bert.sum()} rows wrongly deleted by the cleaning layers")
    assert lost_not_bert.sum() == 0, f"The cleaning layers deleted text containing '{kw}'! Rows: {list(df.index[lost_not_bert][:5])}"

df.to_excel(OUTPUT_DIR + OUTPUT_FILE, index=False)
print(f'Input file is: {INPUT_FILE}')
print(f'Output file is: {OUTPUT_FILE}')

# 🚧 Undetermined: prefix cleaning

In [ ]:
INPUT_DIR = RAW_DIR
OUTPUT_DIR = UNDETERMINED_DIR
INPUT_FILE = '合伙部分.dta'
OUTPUT_FILE = 'undetermined_cleaned.csv'

import re
import pandas as pd

# ===== Step 1: replicate decompose_general_permit.do =====

GENERAL = ["一般经营项目是：","一般经营项目是:","一般经营项目：","一般经营项目:",
           "一般经营项目 ：","一般经营项目 :","一般经营项目（","一般经营项目(",
           "一般项目：","一般项目:","一般项目 ：","一般项目 :",
           "一般事项：","一般事项:",
           "一般经营范围：","一般经营范围:","一般经营范围 ：","一般经营范围 :"]
PERMIT  = ["许可经营项目是：","许可经营项目是:","许可经营项目：","许可经营项目:",
           "许可经营项目 ：","许可经营项目 :","许可经营项目（","许可经营项目(",
           "许可项目：","许可项目:","许可项目 ：","许可项目 :",
           "许可经营范围：","许可经营范围:","许可经营范围 ：","许可经营范围 :"]
DISPLAY = ["自主展示（特色）项目：","自主展示（特色）项目:",
           "自主展示(特色)项目：","自主展示(特色)项目:",
           "自主展示项目：","自主展示项目:",
           "合伙协议记载的经营范围：","合伙协议记载的经营范围:",
           "协议记载的经营范围：","协议记载的经营范围:"]
def first_pos(s, variants):
    for v in variants:
        p = s.find(v)
        if p >= 0:
            return p
    return -1
def decompose(scope):
    """Split scope into [(j, segment)], j: 1=general 2=permit 3=other 4=display.
    Labels stay at the head of their segment (stripped later by clean step)."""
    if pd.isna(scope):
        return []
    s = str(scope)
    hits = []
    for j, vs in [(1, GENERAL), (2, PERMIT), (4, DISPLAY)]:
        p = first_pos(s, vs)
        if p >= 0:
            hits.append((p, j))
    if not hits:
        return [(3, s.strip())]
    hits.sort()
    segs = []
    if hits[0][0] > 0:
        segs.append((3, s[:hits[0][0]].strip()))
    for k, (pos, j) in enumerate(hits):
        end = hits[k+1][0] if k+1 < len(hits) else len(s)
        segs.append((j, s[pos:end].strip()))
    return [(j, t) for j, t in segs if t != '']

# ===== Step 2: replicate clean_label_prefix.do =====

_c1 = re.compile(r'^(一般|许可)(经营)?(项目|事项|范围)(是)?\s*[：:]\s*')
_c2 = re.compile(r'^(一般|许可)(经营)?(项目|事项|范围)(是)?(?=[（(])')
_c3 = re.compile(r'^(合伙)?(协议记载的)?(经营范围|主营业务|主要经营|经营项目|服务)\s*[：:]\s*')
_c3b = re.compile(r'^合伙协议记载的经营范围[:：]\s*')
_c3c = re.compile(r'^章程记载的经营范围[:：]\s*')
_c4 = re.compile(r'^[^：:]{1,18}项目\s*[：:]\s*')

def clean_label_prefix(text):
    s = str(text)
    for pat in (_c1, _c2, _c3, _c3b, _c3c, _c4):
        s = pat.sub('', s, count=1)
    return s.strip()   # Unicode-aware, incl. \u3000 — equivalent to ustrtrim

# ===== Pipeline =====
ID_COLS = ['stock_name', '系统匹配企业名称', '统一社会信用代码', '工商注册号']

df = pd.read_stata(INPUT_DIR + INPUT_FILE, columns=ID_COLS + ['经营范围'])
df = df.rename(columns={'经营范围': 'scope'})          # the firm-name column is no longer renamed
for c in ID_COLS:
    df[c] = df[c].astype(str).str.strip()
df = df.reset_index(drop=True)

# no deduplication: id is the row number in the raw file, one observation per row
df.insert(0, 'row_no', df.index + 1)
df.insert(1, 'id', 'P' + df['row_no'].astype(str))
print(f"Sample: {len(df)} rows (not deduplicated)")

rows = []
for _, r in df.iterrows():
    for seq, (j, seg) in enumerate(decompose(r['scope']), 1):
        rows.append({'row_no': r['row_no'], 'id': r['id'], 'j': j, 'seq': seq,
                     'biz_main': seg, 'scope': r['scope'],
                     **{c: r[c] for c in ID_COLS}})
long = pd.DataFrame(rows).sort_values(['row_no', 'seq'])
long['biz_main_clean'] = long['biz_main'].apply(clean_label_prefix)
long.to_csv(OUTPUT_DIR + OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f'{len(df)} rows -> {len(long)} segments, {len(long)/len(df):.2f} segments per row')
print(f'Input file is: {INPUT_FILE}')
print(f'Output file is: {OUTPUT_FILE}')

# 🚧 Undetermined: suffix cleaning (BERT)

## Load the model

In [ ]:
!pip install transformers datasets -q
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

MODEL_PATH = MODELS_DIR + "cleaned_suffix_model最终清洗suffix的模型"

label_list = ["O", "S"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

# ===== Load the saved model =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded: {MODEL_PATH}, device: {device}")

# ===== Load the labelled data =====
label_data_raw = pd.read_excel(LABELED_FILE)

def build_char_labels(row):
    bizdesc = str(row['bizdesc'])
    bp = str(row['bizdesc_bp']).strip()
    if not bp or bp == 'nan':
        return ['S'] * len(bizdesc)
    matcher = difflib.SequenceMatcher(None, bizdesc, bp)
    labels = ['S'] * len(bizdesc)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            for k in range(i1, i2):
                labels[k] = 'O'
    return labels

# ===== Sliding-window inference =====
def predict_suffix(text, max_len=510, stride=128):
    if len(text) <= max_len:
        enc = tokenizer(text, return_tensors="pt", max_length=512, truncation=True,
                        return_offsets_mapping=True)
        offsets = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()
        char_labels = ['O'] * len(text)
        for tok_idx, (start, end) in enumerate(offsets):
            if start == end:
                continue
            label = id2label[preds[tok_idx]]
            for ci in range(start, end):
                char_labels[ci] = label
    else:
        char_labels = ['O'] * len(text)
        char_scores = [0.0] * len(text)
        starts = list(range(0, len(text) - max_len + 1, stride))
        if starts[-1] + max_len < len(text):
            starts.append(len(text) - max_len)
        for s in starts:
            chunk = text[s:s + max_len]
            enc = tokenizer(chunk, return_tensors="pt", max_length=512, truncation=True,
                            return_offsets_mapping=True)
            offsets = enc.pop("offset_mapping")[0].tolist()
            enc = {k: v.to(model.device) for k, v in enc.items()}
            with torch.no_grad():
                logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[0].cpu()
            preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()
            for tok_idx, (cs, ce) in enumerate(offsets):
                if cs == ce:
                    continue
                confidence = probs[tok_idx].max().item()
                gpos = s + cs
                if gpos < len(text) and confidence > char_scores[gpos]:
                    label = id2label[preds[tok_idx]]
                    for ci in range(gpos, min(s + ce, len(text))):
                        char_labels[ci] = label
                        char_scores[ci] = confidence

    max_gap = 5
    i = 0
    while i < len(char_labels):
        if char_labels[i] == 'S':
            j = i
            while j < len(char_labels) and char_labels[j] == 'S':
                j += 1
            gap_start = j
            k = j
            while k < len(char_labels) and char_labels[k] == 'O':
                k += 1
            gap_len = k - gap_start
            if k < len(char_labels) and char_labels[k] == 'S' and gap_len <= max_gap:
                for p in range(gap_start, k):
                    char_labels[p] = 'S'
            i = j
        else:
            i += 1

    suffix_parts, main_parts = [], []
    current_label, current_text = None, []
    for ch, lb in zip(text, char_labels):
        if lb != current_label:
            if current_label == 'S':
                suffix_parts.append(''.join(current_text))
            current_label = lb
            current_text = [ch]
        else:
            current_text.append(ch)
    if current_label == 'S':
        suffix_parts.append(''.join(current_text))

    current_label, current_text = None, []
    for ch, lb in zip(text, char_labels):
        if lb != current_label:
            if current_label == 'O':
                main_parts.append(''.join(current_text))
            current_label = lb
            current_text = [ch]
        else:
            current_text.append(ch)
    if current_label == 'O':
        main_parts.append(''.join(current_text))

    return {'suffix': ''.join(suffix_parts), 'main': ''.join(main_parts)}

def strip_punct(s):
    return re.sub(r'[。.；;，,、：:！!？?（()）\[\]【】{}「」""''《》\s\^*#〓\n\r]', '', s)

def postprocess_suffix(pred_suffix, original_text):
    open_brackets = '（([【'
    close_brackets = '）)]】'
    close_to_open = {'）':'（', ')':'(', ']':'[', '】':'【'}
    keywords = ['法', '规', '律', '中国', '国家', '须', '批准', '限制', '禁止', '除外',
                '不得', '不含', '除', '凭', '经营活动',
                '无需', '审批', '许可', '市场主体', '备案']

    stack = []
    unmatched_close = []
    for idx, ch in enumerate(pred_suffix):
        if ch in open_brackets:
            stack.append(idx)
        elif ch in close_brackets:
            if stack:
                stack.pop()
            else:
                unmatched_close.append(idx)

    replacements = []
    for idx in unmatched_close:
        close_ch = pred_suffix[idx]
        open_ch = close_to_open.get(close_ch, '（')
        prev_end = 0
        for j in range(idx - 1, -1, -1):
            if pred_suffix[j] in close_brackets:
                prev_end = j + 1
                break
        for length in range(min(8, idx), 1, -1):
            snippet = pred_suffix[idx-length:idx]
            search = snippet + close_ch
            pos = original_text.find(search)
            if pos >= 0:
                actual_close = pos + len(search) - 1
                d = 1
                j = actual_close - 1
                while j >= 0 and d > 0:
                    if original_text[j] == close_ch: d += 1
                    elif original_text[j] == open_ch: d -= 1
                    j -= 1
                if d == 0:
                    full_bracket = original_text[j+1:actual_close+1]
                    replacements.append((prev_end, idx+1, full_bracket))
                break

    fixed = pred_suffix
    for start, end, replacement in sorted(replacements, key=lambda x: x[0], reverse=True):
        fixed = fixed[:start] + replacement + fixed[end:]

    output = []
    i = 0
    current = []
    in_bracket = 0
    while i < len(fixed):
        if fixed[i] in open_brackets:
            if current and in_bracket == 0:
                chunk = ''.join(current)
                if any(k in chunk for k in keywords):
                    output.append(chunk)
                current = []
            in_bracket += 1
            current.append(fixed[i])
        elif fixed[i] in close_brackets:
            current.append(fixed[i])
            in_bracket -= 1
            if in_bracket <= 0:
                in_bracket = 0
                output.append(''.join(current))
                current = []
        else:
            current.append(fixed[i])
        i += 1
    if current:
        chunk = ''.join(current)
        if in_bracket > 0 or any(k in chunk for k in keywords):
            output.append(chunk)
    return ''.join(output)

# ===== Run the test =====
_, test_clean = train_test_split(label_data_raw, test_size=0.1, random_state=42)
test_clean = test_clean.copy()
test_clean['char_labels'] = test_clean.apply(build_char_labels, axis=1)
test_clean = test_clean.rename(columns={'bizdesc': 'scope'})

exact_match = 0
content_match = 0
content_errors = []
total = len(test_clean)

for i, (_, row) in enumerate(test_clean.iterrows()):
    text = str(row['scope'])
    pred = predict_suffix(text)
    pred_suffix = pred['suffix']

    true_parts = []
    cur_label = None
    cur_text = []
    for ch, lb in zip(text, row['char_labels']):
        if lb != cur_label:
            if cur_label == 'S':
                true_parts.append(''.join(cur_text))
            cur_label = lb
            cur_text = [ch]
        else:
            cur_text.append(ch)
    if cur_label == 'S':
        true_parts.append(''.join(cur_text))
    true_suffix = ''.join(true_parts)

    if pred_suffix == true_suffix:
        exact_match += 1
        content_match += 1
    elif postprocess_suffix(pred_suffix, text) == postprocess_suffix(true_suffix, text):
        content_match += 1
        print(f"\n{'='*60}")
        print(f"✅ Row {i} (match after completion)")
        print(f"  S predicted (raw): {pred_suffix}")
        print(f"  S predicted (completed): {postprocess_suffix(pred_suffix, text)}")
        print(f"  S true (raw): {true_suffix}")
        print(f"  S true (completed): {postprocess_suffix(true_suffix, text)}")
    else:
        content_errors.append({
            'row': i,
            'text': text,
            'pred_raw': pred_suffix,
            'true_raw': true_suffix,
            'pred_post': postprocess_suffix(pred_suffix, text),
            'true_post': postprocess_suffix(true_suffix, text),
        })

print(f"\n{'='*60}")
print(f"Exact match: {exact_match}/{total} ({exact_match/total*100:.1f}%)")
print(f"Content match (ignoring punctuation): {content_match}/{total} ({content_match/total*100:.1f}%)")
print(f"Content errors: {len(content_errors)}")

for e in content_errors:
    print(f"\n{'='*60}")
    print(f"❌ Row {e['row']}")
    print(f"  Text: {e['text']}")
    print(f"  S predicted (raw): {e['pred_raw']}")
    print(f"  S predicted (completed): {e['pred_post']}")
    print(f"  S true (raw): {e['true_raw']}")
    print(f"  S true (completed): {e['true_post']}")

## Run BERT on the full undetermined data

In [ ]:
INPUT_DIR = UNDETERMINED_DIR
OUTPUT_DIR = UNDETERMINED_DIR
INPUT_FILE = "undetermined_cleaned.csv"
OUTPUT_FILE = "test_bert_undetermined_cleaned.xlsx"

LABEL_FILE = INPUT_DIR + INPUT_FILE
label_data = pd.read_csv(LABEL_FILE)
all_texts = label_data['biz_main_clean'].astype(str).tolist()
predictions = [predict_suffix(t) for t in tqdm(all_texts)]

label_data['pred_main'] = [p['main'] for p in predictions]
label_data['pred_suffix'] = [p['suffix'] for p in predictions]
label_data['pred_suffix_post'] = [postprocess_suffix(p['suffix'], t) for p, t in zip(predictions, all_texts)]
label_data.to_excel(OUTPUT_DIR + OUTPUT_FILE, index=False)
print(f"Done: {len(label_data)} rows")

print(f'Input file is: {INPUT_FILE}')
print(f'Output file is: {OUTPUT_FILE}')

## Run the post-processing and fixing steps

In [ ]:
INPUT_DIR = UNDETERMINED_DIR
OUTPUT_DIR = UNDETERMINED_DIR
INPUT_FILE = "test_bert_undetermined_cleaned.xlsx"
OUTPUT_FILE = "test_Bert_Undetermined_Post_fix.xlsx"

import re
import pandas as pd
from difflib import SequenceMatcher

# ===== New fixing process: a BERT-faithful layer plus a regex fallback layer (with row-level flags) =====
OPENS  = set('(（[【')
CLOSES = set(')）]】')
NO_CONTENT = re.compile(r'^[。；;，,、\s.．()（）\[\]【】]*$')
DISCLAIMER_KW = re.compile(
    r'依法须经批准|凭营业执照|登记备案后方可|批准后方可开展经营|批准后依批准的内容|'
    r'除许可业务外|市场主体自主选择经营|国务院决定规定|不得以任何方式公开募集|'
    r'无需行政审批|外商投资准入特别管理措施|应取得相关部门许可后方可经营')
INNER_BLOCK = re.compile(r'[（(【\[][^（()）【】\[\]]*[)）\]】]')
BOUNDARY = set('。；;，,、.．：:！!？?＂"“” \t^〓*＊#＃=＝※') | OPENS | CLOSES

BRACKET_NUM = re.compile(r'[(（]\s*(?:[0-9]{1,2}|[一二三四五六七八九十]{1,3})\s*[)）]')
SEG_NUM = re.compile(r'(?:^|(?<=[。；;，,、]))\s*(?:[0-9]{1,2}|[一二三四五六七八九十]{1,3})\s*(?:[、::：]|[..](?![0-9]))')
SEG_LABEL = re.compile(r'(?:^|(?<=[。；;，,]))(?:许可经营项目|一般经营项目|许可项目|一般项目|经营范围)(?:是)?[:：]?'
                       r'|(?:^|(?<=[。；;，,]))(?:服务|资本市场服务)(?:从)?[:：]')
DATE = r'\d{4}年\d{1,2}月\d{1,2}日'
DURATION = re.compile(
    rf'(?:合伙|营业|经营|存续)(?:企业)?期限[:：]?(?:为|是)?'
    rf'(?:自(?:{DATE}|设立之日)起?)?'
    rf'(?:至{DATE}|[一二三四五六七八九十百\d]+年|长期)?止?')

def clean_bcd(s):
    s = BRACKET_NUM.sub('；', s)
    s = SEG_LABEL.sub('', s)
    s = SEG_NUM.sub('', s)
    s = SEG_LABEL.sub('', s)     # numbers and labels can hide each other: strip each once, then one more round
    s = _drop_orphan_brackets(s)
    s = DURATION.sub('', s)
    return s

def split_suffix_pieces(s):
    pieces, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch in OPENS:
            if depth == 0 and i > start:
                pieces.append(s[start:i]); start = i
            depth += 1
        elif ch in CLOSES:
            if depth > 0:
                depth -= 1
                if depth == 0:
                    pieces.append(s[start:i+1]); start = i + 1
    if start < len(s):
        pieces.append(s[start:])
    return pieces

def _balanced(s):
    st = []
    for ch in s:
        if ch in OPENS: st.append(ch)
        elif ch in CLOSES:
            if not st: return False
            st.pop()
    return not st

def _orphan_idx(s):
    bad, st = set(), []
    for i, ch in enumerate(s):
        if ch in OPENS: st.append(i)
        elif ch in CLOSES:
            if st: st.pop()
            else: bad.add(i)
    bad.update(st)
    return bad

def _drop_orphan_brackets(s):
    bad = _orphan_idx(s)
    return ''.join(ch for i, ch in enumerate(s) if i not in bad)

def sweep_disclaimers_count(text):
    """Regex sweep as the last layer, with a count: returns (cleaned text, number of boilerplate blocks removed)"""
    n = 0
    prev = None
    while prev != text:
        prev = text
        for m in INNER_BLOCK.finditer(text):
            if DISCLAIMER_KW.search(m.group()):
                text = text[:m.start()] + text[m.end():]
                n += 1
                break
    return text, n

def tidy_punct(s):
    """Tidy the punctuation left after deletions"""
    s = re.sub(r'[\^〓＊*＃#＝=※]+', '；', s)              # garbled separators -> ；
    s = re.sub(r'([（(【\[])\s*[。；;，,、.]+', r'\1', s)    # punctuation right inside an opening bracket
    s = re.sub(r'[。；;，,、.]+\s*([)）\]】])', r'\1', s)    # punctuation right inside a closing bracket
    def _collapse(m):                                       # collapse repeated punctuation, keeping the strongest mark
        run = m.group()
        if '。' in run or '.' in run: return '。'
        if '；' in run or ';' in run: return '；'
        if '，' in run or ',' in run: return '，'
        if '、' in run: return '、'
        return ''                                           # whitespace only
    s = re.sub(r'[。；;，,、.\s]{2,}', _collapse, s)
    s = s.lstrip('。；;，,、. ')                             # strip leading punctuation
    if s and s[-1] in '；;，,、.':
        s = s[:-1] + '。'                                   # end with a full stop
    s = re.sub(r'（）|\(\)|【】|\[\]', '', s)                # empty bracket pairs
    return s.strip()

def remove_suffixes_v6_flagged(text, suffix_str, min_span=3, min_cover=0.8):
    """v6 with row-level flags. Returns a Series:
       post      cleaned text
       n_exact   blocks deleted exactly as a whole (layer 1, BERT labels applied as they are)
       n_align   phantom blocks handled by sequence alignment (layer 2)
       n_giveup  phantom blocks skipped for low alignment coverage (better to under-delete)
       n_sweep   boilerplate blocks removed by the regex sweep (layer 3)"""
    if pd.isna(text):
        return pd.Series([text, 0, 0, 0, 0],
                         index=['post', 'n_exact', 'n_align', 'n_giveup', 'n_sweep'])
    r = str(text)
    n_exact = n_align = n_giveup = 0
    if not (pd.isna(suffix_str) or not str(suffix_str).strip()):
        failed = []
        for p in split_suffix_pieces(str(suffix_str)):
            if NO_CONTENT.match(p):
                continue
            c = p.strip('。；;，,、 ')
            if not _balanced(p) or (not any(ch in OPENS | CLOSES for ch in p)  # added 7/14 to avoid deleting suffix segments that are too long
                                    and len(c) < 6):        #
                failed.append(p); continue                        #
            if p in r:
                r = r.replace(p, '', 1); n_exact += 1
            elif c in r and len(c) >= 4:
                r = r.replace(c, '', 1); n_exact += 1
            else:
                failed.append(p)
        for p in failed:
            sm = SequenceMatcher(None, p, r, autojunk=False)
            blocks = [b for b in sm.get_matching_blocks() if b.size >= min_span]
            if sum(b.size for b in blocks) < min_cover * len(p):
                n_giveup += 1
                continue
            n_align += 1
            for b in sorted(blocks, key=lambda b: b.b, reverse=True):
                s0, s1 = b.b, b.b + b.size
                span = r[s0:s1]
                bad = _orphan_idx(span)
                while bad and (0 in bad or len(span) - 1 in bad):
                    if 0 in bad: s0 += 1
                    else: s1 -= 1
                    span = r[s0:s1]; bad = _orphan_idx(span)
                if len(span) < min_span or not _balanced(span):
                    continue
                if len(span) < 10:      # short span: both ends must fall on natural boundaries, so no characters are cut from inside a word
                    left_ok  = s0 == 0 or r[s0-1] in BOUNDARY or span[0] in OPENS
                    right_ok = s1 >= len(r) or r[s1] in BOUNDARY or span[-1] in CLOSES
                    if not (left_ok and right_ok):
                        continue
                r = r[:s0] + r[s1:]
        if failed:
            r = _drop_orphan_brackets(r)
    r, n_sweep = sweep_disclaimers_count(r)
    return pd.Series([tidy_punct(clean_bcd(r)), n_exact, n_align, n_giveup, n_sweep],
                     index=['post', 'n_exact', 'n_align', 'n_giveup', 'n_sweep'])

# ===== Pipeline =====
df = pd.read_excel(INPUT_DIR + INPUT_FILE)
print(f"Read {len(df)} rows")

old_post = df['post_biz_main_clean'].fillna('').astype(str) \
           if 'post_biz_main_clean' in df.columns else None

res = df.apply(lambda r: remove_suffixes_v6_flagged(
    r['biz_main_clean'], r['pred_suffix_post']), axis=1)
df['post_biz_main_clean'] = res['post']
df['fix_n_exact']  = res['n_exact'].astype(int)
df['fix_n_align']  = res['n_align'].astype(int)     # >0 = a phantom block in this row was realigned
df['fix_n_giveup'] = res['n_giveup'].astype(int)    # >0 = a suffix block in this row was skipped (may remain)
df['fix_n_sweep']  = res['n_sweep'].astype(int)     # >0 = the regex fallback was used

# ===== Change statistics =====
print("=" * 60)
print(f"Layer 1, exact block deletion: {(df['fix_n_exact'] > 0).sum()} rows, {df['fix_n_exact'].sum()} blocks")
print(f"Layer 2, phantom blocks aligned: {(df['fix_n_align'] > 0).sum()} rows, {df['fix_n_align'].sum()} blocks")
print(f"         phantom blocks skipped: {(df['fix_n_giveup'] > 0).sum()} rows, {df['fix_n_giveup'].sum()} blocks (suffix may remain)")
print(f"Layer 3, regex sweep of boilerplate: {(df['fix_n_sweep'] > 0).sum()} rows, {df['fix_n_sweep'].sum()} blocks")

new_post = df['post_biz_main_clean'].fillna('').astype(str)
if old_post is not None:
    changed = (old_post != new_post)
    print(f"Rows that differ from the old post_biz_main_clean: {changed.sum()} / {len(df)}")
    for _, row in df[changed].head(10).iterrows():
        print("=" * 60)
        print(f"  suffix:  {row['pred_suffix_post']}")
        print(f"  old clean: {old_post[row.name]}")
        print(f"  new clean: {row['post_biz_main_clean']}")

# ===== Self-check: brackets and boilerplate =====
def _bal_bad(s):
    st, uc = [], 0
    for ch in str(s):
        if ch in OPENS: st.append(ch)
        elif ch in CLOSES:
            if st: st.pop()
            else: uc += 1
    return len(st) + uc > 0

print("=" * 60)
print(f"Rows with unbalanced brackets: {new_post.apply(_bal_bad).sum()} (already broken in the raw text: "
      f"{df['biz_main_clean'].fillna('').astype(str).apply(_bal_bad).sum()})")
print(f"Rows still containing '依法须经批准' (subject to approval by law): {new_post.str.contains('依法须经批准').sum()}")

# ===== Deletion audit: warn when cleaning removed text that BERT did not label =====
raw = df['biz_main_clean'].fillna('').astype(str)
sfx = df['pred_suffix_post'].fillna('').astype(str)
for kw in ['股权投资', '创业投资', '投资管理', '投资咨询', '资产管理', '基金管理']:
    lost_not_bert = (raw.str.contains(kw, regex=False)
                     & ~new_post.str.contains(kw, regex=False)
                     & ~sfx.str.contains(kw, regex=False))
    status = "OK" if lost_not_bert.sum() == 0 else "!!!"
    print(f"Audit [{status}] '{kw}': {lost_not_bert.sum()} rows wrongly deleted by the cleaning layers")
    assert lost_not_bert.sum() == 0, f"The cleaning layers deleted text containing '{kw}'! Rows: {list(df.index[lost_not_bert][:5])}"

df.to_excel(OUTPUT_DIR + OUTPUT_FILE, index=False)
print(f'Input file is: {INPUT_FILE}')
print(f'Output file is: {OUTPUT_FILE}')

# Benchmark: first screen for core statements

Using `onion peeling` until the variance converges

In [ ]:
!pip install -q sentence-transformers pandas

from google.colab import drive
drive.mount('/content/drive')

import re, os, time, gc
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

INPUT_DIR = CORE_DIR
OUTPUT_DIR = CORE_DIR
EMB_CACHE_DIR = MODELS_DIR
INPUT_FILE = "test_Bert_Benchmark_Post_fix.xlsx"
OUTPUT_FILE = "0.03_test_4.18 Iterative Similarity(2).csv"

TEXT_COL = "post_biz_main_clean"
MIN_LEN = 2

# ===== Read in and keep fund partnerships =====
df_raw = pd.read_excel(INPUT_DIR + INPUT_FILE)
print(f"Read {len(df_raw)} rows")
print("Columns:", df_raw.columns.tolist())

df_raw[TEXT_COL] = df_raw[TEXT_COL].fillna("").astype(str)
df_raw["indicator"] = df_raw["indicator"].fillna("").astype(str)
df_raw["原文件导入名称"] = df_raw["原文件导入名称"].fillna("").astype(str)

mask_fund = df_raw["indicator"].str.contains("fund", case=False, regex=True)
mask_partner = df_raw["原文件导入名称"].str.contains("合伙企业|有限合伙", regex=True)
df_fund = df_raw[mask_fund & mask_partner].reset_index(drop=True)

print(f"indicator contains fund:      {mask_fund.sum()}")
print(f"name contains partnership:    {mask_partner.sum()}")
print(f"both:                         {len(df_fund)}")


def clean_biz(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"^\d+[、\.:：\)）]\s*", "", s)
    s = re.sub(r"^[^\w\u4e00-\u9fff（(【\[]+|[^\w\u4e00-\u9fff）)】\]]+$", "", s)
    s = re.sub(r"\d+[、\.:：]\s*", "", s)
    s = re.sub(r"^[\(（]?\d+[、\.:：\)）]\s*", "", s)
    s = re.sub(r"[^\w\u4e00-\u9fff）)】\]]*\s*无\s*$", "", s)
    s = re.sub(r"^无[\s、，,；;]+", "", s)
    return s.strip()


df_fund[TEXT_COL] = df_fund[TEXT_COL].apply(clean_biz)
df_fund["text_len"] = df_fund[TEXT_COL].str.len()

before = len(df_fund)
df_valid = df_fund[df_fund["text_len"] >= MIN_LEN].reset_index(drop=True)
df_invalid = df_fund[df_fund["text_len"] < MIN_LEN].reset_index(drop=True)
print(f"Symbol cleaning + filter len>={MIN_LEN}: {before} → {len(df_valid)}")
print(f"Valid: {len(df_valid)}  |  dropped as too short: {len(df_invalid)}")

texts = df_valid[TEXT_COL].tolist()
N = len(texts)

# ===== Step 1: bge-m3 embeddings =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

model = SentenceTransformer("BAAI/bge-m3", device=device)

t0 = time.time()
emb = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)
print(f"Encoding done: {emb.shape}, {(time.time()-t0)/60:.1f} min")

np.save(EMB_CACHE_DIR + EMB_CACHE_FILE, emb)
print(f"Embeddings cached: {EMB_CACHE_FILE} ({emb.nbytes/1e6:.0f} MB)")

del model
gc.collect()
torch.cuda.empty_cache()

# ===== Step 2: outlier detection by iterative peeling =====
print("\n" + "=" * 60)
print("Start iterative peeling")
print("=" * 60)

TRIM_RATIO = 0.01 #0.01
MAX_ROUNDS = 10  #10
VAR_TOL = 0.03  # 0.05 and 0.03 both work well
PATIENCE = 2  #2
CHUNK = 4096  #4096

emb_gpu = torch.from_numpy(emb).to(device)
round_ejected = np.full(N, -1, dtype=np.int32)
core_idx = np.arange(N)
score_history = np.full((N, MAX_ROUNDS), np.nan, dtype=np.float32)
history = []
patience_count = 0
prev_var = None

for round_i in range(MAX_ROUNDS):
    t0 = time.time()
    M = len(core_idx)
    print(f"\n===== Round {round_i} | core pool: {M} =====")

    if M < 100:
        print("Core pool too small, stop")
        break

    core_emb = emb_gpu[core_idx]

    core_scores = torch.zeros(M, device=device)
    for start in range(0, M, CHUNK):
        end = min(start + CHUNK, M)
        block = core_emb[start:end]
        sims = block @ core_emb.T
        rows_local = torch.arange(end - start, device=device)
        rows_global = torch.arange(start, end, device=device)
        sims[rows_local, rows_global] = float("nan")
        core_scores[start:end] = torch.nanmean(sims, dim=1)
    core_scores_np = core_scores.cpu().numpy()

    ejected_idx = np.where(round_ejected != -1)[0]
    if len(ejected_idx) > 0:
        ejected_emb = emb_gpu[ejected_idx]
        ejected_scores = torch.zeros(len(ejected_idx), device=device)
        for start in range(0, len(ejected_idx), CHUNK):
            end = min(start + CHUNK, len(ejected_idx))
            block = ejected_emb[start:end]
            sims = block @ core_emb.T
            ejected_scores[start:end] = sims.mean(dim=1)
        ejected_scores_np = ejected_scores.cpu().numpy()
    else:
        ejected_scores_np = np.array([])

    score_history[core_idx, round_i] = core_scores_np
    if len(ejected_idx) > 0:
        score_history[ejected_idx, round_i] = ejected_scores_np

    mean_v = float(core_scores_np.mean())
    var_v = float(core_scores_np.var())
    std_v = float(core_scores_np.std())
    history.append({
        "round": round_i, "core_size": M,
        "mean": mean_v, "variance": var_v, "std": std_v,
    })
    print(f"  mean={mean_v:.4f}  variance={var_v:.6f}  std={std_v:.4f}  time {time.time()-t0:.1f}s")

    if prev_var is not None:
        rel_change = abs(var_v - prev_var) / prev_var if prev_var > 0 else 0
        print(f"  relative change in variance: {rel_change*100:.2f}%")
        if rel_change < VAR_TOL:
            patience_count += 1
            print(f"  ✓ convergence met in {patience_count}/{PATIENCE} consecutive rounds")
            if patience_count >= PATIENCE:
                print(f"\n✅ Converged, total rounds = {round_i + 1}")
                break
        else:
            patience_count = 0
    prev_var = var_v

    n_trim = int(M * TRIM_RATIO)
    if n_trim == 0:
        break
    lowest_order = np.argsort(core_scores_np)[:n_trim]
    ejected_this_round = core_idx[lowest_order]
    round_ejected[ejected_this_round] = round_i
    keep_mask = np.ones(M, dtype=bool)
    keep_mask[lowest_order] = False
    core_idx = core_idx[keep_mask]
    print(f"  Removed this round: {n_trim}, new core pool = {len(core_idx)}")

else:
    print(f"\n⚠️ Reached the maximum of {MAX_ROUNDS} rounds without converging")

del emb_gpu
gc.collect()
torch.cuda.empty_cache()

# ===== Step 3: collect and save the results =====
n_rounds_actual = len(history)
df_history = pd.DataFrame(history)
print("\n" + "=" * 60)
print("Trajectory by round:")
print("=" * 60)
print(df_history.to_string(index=False))

df_valid["round_ejected"] = round_ejected
df_valid["still_in_core"] = (round_ejected == -1).astype(int)
for r in range(n_rounds_actual):
    df_valid[f"score_r{r}"] = score_history[:, r]
df_valid["first_round_score"] = score_history[:, 0]
df_valid["final_round_score"] = score_history[:, n_rounds_actual - 1]

rounds_vec = np.arange(n_rounds_actual)
slopes = np.full(N, np.nan, dtype=np.float32)
for i in range(N):
    y = score_history[i, :n_rounds_actual]
    mask = ~np.isnan(y)
    if mask.sum() >= 2:
        slopes[i] = np.polyfit(rounds_vec[mask], y[mask], 1)[0]
df_valid["trajectory_slope"] = slopes

df_valid["_key"] = np.where(
    df_valid["round_ejected"] == -1,
    99999 - df_valid["final_round_score"] * 1000,
    df_valid["round_ejected"] - df_valid["first_round_score"],
)
df_out = df_valid.sort_values("_key").reset_index(drop=True).drop(columns="_key")
df_out["outlier_rank"] = range(1, len(df_out) + 1)

df_out.to_csv(OUTPUT_DIR + OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("\n" + "=" * 70)
print("The 15 largest outliers:")
print("=" * 70)
for _, row in df_out.head(15).iterrows():
    ejected = int(row["round_ejected"])
    tag = f"removed in round {ejected}" if ejected >= 0 else "core pool"
    print(f"\n[rank{row['outlier_rank']}] {tag}  "
          f"r0={row['first_round_score']:.3f}  "
          f"final={row['final_round_score']:.3f}  "
          f"slope={row['trajectory_slope']:+.4f}")
    print(f"  {row[TEXT_COL][:180]}")

print(f"\nInput file is: {INPUT_FILE}")
print(f"Output file is: {OUTPUT_FILE}")
print(f"Embedding cache is: {EMB_CACHE_FILE}")

## Comparison with LOF

In [ ]:
!pip install -q sentence-transformers pandas

from google.colab import drive
drive.mount('/content/drive')

import re, os, time, gc
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import LocalOutlierFactor

# ===== Paths and parameters =====
INPUT_DIR  = CORE_DIR
OUTPUT_DIR = CORE_DIR
EMB_CACHE_DIR = MODELS_DIR

INPUT_FILE     = "test_Bert_Benchmark_Post_fix.xlsx"
OUTPUT_FILE    = "test_4.18 LOF.csv"
EMB_CACHE_FILE = "0.03_test_emb_bge_m3.npy"   # same embedding cache as the peeling step

TEXT_COL    = "post_biz_main_clean"
MIN_LEN     = 2                                # same definition as the peeling test version
N_NEIGHBORS = 80                               # original parameters

# ===== Read in and filter (identical to peeling) =====
df_raw = pd.read_excel(INPUT_DIR + INPUT_FILE)
print(f"Read {len(df_raw)} rows")
df_raw[TEXT_COL] = df_raw[TEXT_COL].fillna("").astype(str)
df_raw["indicator"] = df_raw["indicator"].fillna("").astype(str)
df_raw["原文件导入名称"] = df_raw["原文件导入名称"].fillna("").astype(str)

mask_fund    = df_raw["indicator"].str.contains("fund", case=False, regex=True)
mask_partner = df_raw["原文件导入名称"].str.contains("合伙企业|有限合伙", regex=True)
df_fund = df_raw[mask_fund & mask_partner].reset_index(drop=True)
print(f"fund + partnership: {len(df_fund)}")

def clean_biz(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"^\d+[、\.:：\)）]\s*", "", s)
    s = re.sub(r"^[^\w\u4e00-\u9fff（(【\[]+|[^\w\u4e00-\u9fff）)】\]]+$", "", s)
    s = re.sub(r"\d+[、\.:：]\s*", "", s)
    s = re.sub(r"^[\(（]?\d+[、\.:：\)）]\s*", "", s)
    s = re.sub(r"[^\w\u4e00-\u9fff）)】\]]*\s*无\s*$", "", s)
    s = re.sub(r"^无[\s、，,；;]+", "", s)
    return s.strip()

df_fund[TEXT_COL] = df_fund[TEXT_COL].apply(clean_biz)
df_fund["text_len"] = df_fund[TEXT_COL].str.len()
before = len(df_fund)
df_valid = df_fund[df_fund["text_len"] >= MIN_LEN].reset_index(drop=True)
print(f"Cleaning + filter len>={MIN_LEN}: {before} → {len(df_valid)}")

texts = df_valid[TEXT_COL].tolist()
N = len(texts)

# ===== Embeddings (load the cache if the row count matches) =====
cache_path = EMB_CACHE_DIR + EMB_CACHE_FILE
emb = None
if os.path.exists(cache_path):
    cached = np.load(cache_path)
    if len(cached) == N:
        emb = cached; print(f"Loaded cached embeddings: {emb.shape}")
    else:
        print(f"Cache has {len(cached)} rows, not {N}: re-encoding")
if emb is None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer("BAAI/bge-m3", device=device)
    emb = model.encode(texts, batch_size=64, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    np.save(cache_path, emb)
    del model; gc.collect(); torch.cuda.empty_cache()

assert emb.shape[0] == len(df_valid)

# ===== LOF (original settings: n_neighbors=80, euclidean, contamination=0.05) =====
print(f"\nRunning LOF (n_neighbors={N_NEIGHBORS})...")
t0 = time.time()
lof = LocalOutlierFactor(n_neighbors=N_NEIGHBORS, metric="euclidean",
                         contamination=0.05, n_jobs=-1)
labels = lof.fit_predict(emb)                 # +1 inlier / -1 outlier (auto => lof_score > 1.5)
lof_score = -lof.negative_outlier_factor_
print(f"LOF done, {time.time()-t0:.1f}s")
print(f"Flagged as outliers: {(labels == -1).sum()} ({(labels == -1).mean()*100:.1f}%)")

# ===== Collect the results and compare with peeling =====
df_valid["lof_score"]   = lof_score
df_valid["lof_outlier"] = (labels == -1).astype(int)     # 1 = outlier (original cut: score > 1.5)
df_out = df_valid.sort_values("lof_score", ascending=False).reset_index(drop=True)
df_out["lof_rank"] = range(1, len(df_out) + 1)
df_out.to_csv(OUTPUT_DIR + OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_DIR + OUTPUT_FILE)

PEELING_FILE = "0.03_test_4.18 Iterative Similarity(2).csv"
if os.path.exists(OUTPUT_DIR + PEELING_FILE):
    df_peel = pd.read_csv(OUTPUT_DIR + PEELING_FILE)
    cmp = df_out[["id", "lof_score", "lof_outlier"]].merge(
        df_peel[["id", "still_in_core", "round_ejected"]], on="id", how="inner")
    cmp["lof_core"] = 1 - cmp["lof_outlier"]
    print("\nThe two methods, 2x2 table:")
    print(pd.crosstab(cmp["still_in_core"], cmp["lof_core"],
                      rownames=["peeling core"], colnames=["LOF core"]))
    print(f"Agreement: {(cmp['still_in_core']==cmp['lof_core']).mean()*100:.1f}%")

print("\nThe 10 largest LOF outliers:")
for _, row in df_out.head(10).iterrows():
    print(f"[rank{row['lof_rank']}] lof={row['lof_score']:.3f}  {row[TEXT_COL][:120]}")

## LOF tuning

In [ ]:
!pip install -q sentence-transformers pandas

from google.colab import drive
drive.mount('/content/drive')

import re, os, time, gc
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import LocalOutlierFactor

# ===== Paths and parameters =====
INPUT_DIR  = CORE_DIR
OUTPUT_DIR = CORE_DIR
EMB_CACHE_DIR = MODELS_DIR

INPUT_FILE     = "test_Bert_Benchmark_Post_fix.xlsx"
OUTPUT_FILE    = "ratio.csv"
EMB_CACHE_FILE = "0.03_test_emb_bge_m3.npy"   # same embedding cache as the peeling step

TEXT_COL    = "post_biz_main_clean"
MIN_LEN     = 2                                # same definition as the peeling test version
N_NEIGHBORS = [1, 5, 10, 20, 50, 100, 200, 500, 1000]
METRICS     = ["euclidean", "cosine"]  # minkowski (p=2) equals euclidean, so it is dropped
CONTAMINATIONS = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, "auto"]

# ===== Read in and filter (identical to peeling) =====
df_raw = pd.read_excel(INPUT_DIR + INPUT_FILE)
print(f"Read {len(df_raw)} rows")
df_raw[TEXT_COL] = df_raw[TEXT_COL].fillna("").astype(str)
df_raw["indicator"] = df_raw["indicator"].fillna("").astype(str)
df_raw["原文件导入名称"] = df_raw["原文件导入名称"].fillna("").astype(str)

mask_fund    = df_raw["indicator"].str.contains("fund", case=False, regex=True)
mask_partner = df_raw["原文件导入名称"].str.contains("合伙企业|有限合伙", regex=True)
df_fund = df_raw[mask_fund & mask_partner].reset_index(drop=True)
print(f"fund + partnership: {len(df_fund)}")

def clean_biz(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"^\d+[、\.:：\)）]\s*", "", s)
    s = re.sub(r"^[^\w\u4e00-\u9fff（(【\[]+|[^\w\u4e00-\u9fff）)】\]]+$", "", s)
    s = re.sub(r"\d+[、\.:：]\s*", "", s)
    s = re.sub(r"^[\(（]?\d+[、\.:：\)）]\s*", "", s)
    s = re.sub(r"[^\w\u4e00-\u9fff）)】\]]*\s*无\s*$", "", s)
    s = re.sub(r"^无[\s、，,；;]+", "", s)
    return s.strip()

def gq_flag(text):
    """1 if the text contains '股权投资' (equity investment) or '创业投资' (venture capital), else 0 (row-level flag)"""
    return int(bool(re.search(r'股权投资|创业投资', str(text))))

df_fund[TEXT_COL] = df_fund[TEXT_COL].apply(clean_biz)
df_fund["text_len"] = df_fund[TEXT_COL].str.len()
before = len(df_fund)
df_valid = df_fund[df_fund["text_len"] >= MIN_LEN].reset_index(drop=True)
print(f"Cleaning + filter len>={MIN_LEN}: {before} → {len(df_valid)}")

# compute the gq flag once
df_valid["gq_flag"] = df_valid[TEXT_COL].apply(gq_flag)
print(f"Rows mentioning equity or venture investment: {df_valid['gq_flag'].sum()} "
      f"({df_valid['gq_flag'].mean()*100:.1f}%)")

texts = df_valid[TEXT_COL].tolist()
N = len(texts)

# ===== Embeddings (load the cache if the row count matches) =====
cache_path = EMB_CACHE_DIR + EMB_CACHE_FILE
emb = None
if os.path.exists(cache_path):
    cached = np.load(cache_path)
    if len(cached) == N:
        emb = cached; print(f"Loaded cached embeddings: {emb.shape}")
    else:
        print(f"Cache has {len(cached)} rows, not {N}: re-encoding")
if emb is None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer("BAAI/bge-m3", device=device)
    emb = model.encode(texts, batch_size=64, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    np.save(cache_path, emb)
    del model; gc.collect(); torch.cuda.empty_cache()

assert emb.shape[0] == len(df_valid)

# ===== LOF grid search =====
# Key speed-up: contamination does not change the LOF score, only the threshold.
# Fit once per (n_neighbors, metric), then sweep every contamination over the scores.
records = []
total = len(N_NEIGHBORS) * len(METRICS)
done = 0

for n in N_NEIGHBORS:
    for m in METRICS:
        done += 1
        t0 = time.time()
        lof_model = LocalOutlierFactor(n_neighbors=n, metric=m, n_jobs=-1)
        lof_model.fit(emb)
        nof = lof_model.negative_outlier_factor_       # smaller = more outlying
        print(f"[{done}/{total}] n_neighbors={n}, metric={m}, "
              f"fit time {time.time()-t0:.1f}s")

        for c in CONTAMINATIONS:
            # reproduce sklearn's internal threshold rule
            if c == "auto":
                offset = -1.5
            else:
                offset = np.percentile(nof, 100.0 * c)
            outlier = nof < offset                     # True = outlier

            n_out = int(outlier.sum())
            if n_out == 0:
                ratio = np.nan
            else:
                ratio = df_valid.loc[outlier, "gq_flag"].sum() / n_out

            records.append({
                "n_neighbors": n,
                "metric": m,
                "contamination": c,
                "n_outlier": n_out,
                "outlier_pct": round(n_out / N * 100, 2),
                "ratio": ratio,
            })

res = pd.DataFrame(records)
res.to_csv(OUTPUT_DIR + OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\nDone: {len(res)} parameter sets → {OUTPUT_DIR + OUTPUT_FILE}")
res.sort_values("ratio").head(20)

## Figure 1

In [ ]:
import pandas as pd, numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go

EMB_FILE  = MODELS_DIR + "0.03_test_emb_bge_m3.npy"
PEEL_FILE = CORE_DIR + "0.03_test_4.18 Iterative Similarity(2).csv"

emb = np.load(EMB_FILE)
df  = pd.read_csv(PEEL_FILE).sort_values(["id", "j"]).reset_index(drop=True)
assert len(df) == emb.shape[0]

pca = PCA(n_components=3, random_state=0)
xyz = pca.fit_transform(emb)
print("Variance explained by PC1+PC2+PC3: %.1f%%" % (100 * pca.explained_variance_ratio_.sum()))
df["pc1"], df["pc2"], df["pc3"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
core = df["still_in_core"] == 1

# too many points make the interactive plot lag: sample 20,000 grey points, keep all red ones
grey = df[core].sample(min(20000, core.sum()), random_state=0)
red  = df[~core]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=grey["pc1"], y=grey["pc2"], z=grey["pc3"],
    mode="markers", name=f"core (sample {len(grey)}/{core.sum()})",
    marker=dict(size=1.5, color="grey", opacity=0.25)))
fig.add_trace(go.Scatter3d(
    x=red["pc1"], y=red["pc2"], z=red["pc3"],
    mode="markers", name=f"ejected ({len(red)})",
    marker=dict(size=2.5, color="crimson", opacity=0.6),
    text=red["post_biz_main_clean"].str[:60]))     # hover shows the first 60 characters
fig.update_layout(scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
                  width=900, height=700, legend=dict(x=0, y=1))
fig.show()

In [ ]:
import pandas as pd, numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go

EMB_FILE = MODELS_DIR + "0.03_test_emb_bge_m3.npy"
LOF_FILE = CORE_DIR + "test_4.18 LOF.csv"

emb = np.load(EMB_FILE)
df  = pd.read_csv(LOF_FILE).sort_values(["id", "j"]).reset_index(drop=True)   # restore the embedding row order
assert len(df) == emb.shape[0], f"{len(df)} vs {emb.shape[0]} — LOF and emb do not come from the same preprocessing run"

pca = PCA(n_components=3, random_state=0)
xyz = pca.fit_transform(emb)
print("Variance explained by PC1+PC2+PC3: %.1f%%" % (100 * pca.explained_variance_ratio_.sum()))
df["pc1"], df["pc2"], df["pc3"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]

out = df["lof_outlier"] == 1
grey = df[~out].sample(min(20000, (~out).sum()), random_state=0)
red  = df[out]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=grey["pc1"], y=grey["pc2"], z=grey["pc3"],
    mode="markers", name=f"normal (sample {len(grey)}/{(~out).sum()})",
    marker=dict(size=1.5, color="grey", opacity=0.25)))
fig.add_trace(go.Scatter3d(
    x=red["pc1"], y=red["pc2"], z=red["pc3"],
    mode="markers", name=f"LOF outlier ({out.sum()})",
    marker=dict(size=2.5, color="royalblue", opacity=0.65),
    text=red["post_biz_main_clean"].str[:60]))
fig.update_layout(scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
                  width=900, height=700, legend=dict(x=0, y=1),
                  title="PCA 3D — LOF outliers")
fig.show()

# Add the hand-screened core.dta

# Position statistics of core items

## Masked version

In [ ]:
import re
import pandas as pd
INPUT_DIR = CORE_DIR
OUTPUT_DIR = INTERMEDIATE_DIR + "Results/"
INPUT_FILE = '0.03_test_4.18 Iterative Similarity(2).csv'
OUTPUT_ALLITEMS = 'in_core_aftermask_allitems_with_pos.csv'  # when rerunning, confirm the filter first
OUTPUT_BYPOS = 'in_core_aftermask_allitems_by_position.csv'

MASK_FILE_SPECIAL = pd.read_excel(HUMANLY_PICKED_DIR + '1. item split results.xlsx')
MASK_FILE_NORMAL  = pd.read_stata(HUMANLY_PICKED_DIR + '1. 按顿号split.dta')
# split by the hand-made special map
SPECIAL_MAP = (MASK_FILE_SPECIAL
               .dropna(subset=['item', 'split'])
               .astype({'item': str, 'split': str})
               .groupby('item', sort=False)['split']
               .apply(list)
               .to_dict())
# split on enumeration commas (、): NORMAL_SET
NORMAL_SET = set(MASK_FILE_NORMAL['item'].dropna().astype(str))

print(f"special covers {len(SPECIAL_MAP)}, normal split on the enumeration comma: {len(NORMAL_SET)}")

def apply_mask(seg):
    if seg in SPECIAL_MAP:
        return SPECIAL_MAP[seg]
    if seg in NORMAL_SET:
        return [p.strip() for p in seg.split('、') if p.strip()]
    return [seg]

# ===== Bracket-aware sentence splitting =====
OPENS  = set('(（[【')
CLOSES = set(')）]】')
NUM_PREFIX = re.compile(r"^[0-9一二三四五六七八九十]+[..、]")
ITEM_STOP = {
    "", "无", "的", "依法须经批准的项目", "经相关部门批准后方可开展经营活动",
    "不含许可经营项目", "国家有专项规定的除外",
}
JUNK_STRIP = ' ^〓*#"\'`~·-—_=|'
DISCLAIMER_KW = re.compile(
    r'依法须经批准|凭营业执照|登记备案后方可|批准后方可开展经营|批准后依批准的内容|'
    r'除许可业务外|市场主体自主选择经营|国务院决定规定|不得以任何方式公开募集|'
    r'无需行政审批|外商投资准入特别管理措施|应取得相关部门许可后方可经营')
MAIN_SPLIT = set(';；,，。.^〓')
FALLBACK_SPLIT = set()      #set('、')

def _wrapped(t):
    """Whether the whole segment is wrapped in one pair of brackets"""
    if not t or t[0] not in OPENS:
        return False
    depth = 0
    for i, ch in enumerate(t):
        if ch in OPENS:
            depth += 1
        elif ch in CLOSES:
            depth -= 1
            if depth == 0:
                return i == len(t) - 1
    return False

def explode_items(df, text_col, max_depth=3):
    """Bracket-aware splitting: no split at depth > 0; a segment wrapped in brackets is unwrapped and split again, recursively;
       boilerplate fragments are filtered by keyword at the item level. Output columns (fid, item, pos, pos_bin)."""
    rows = []
    def _segs(t):
      # decide which separator to split this segment on
      split_set = MAIN_SPLIT if any(ch in MAIN_SPLIT for ch in t) else FALLBACK_SPLIT
      segs, depth, start = [], 0, 0
      for i, ch in enumerate(t):
          if ch in OPENS:
              depth += 1
          elif ch in CLOSES:
              depth = max(0, depth - 1)
          elif ch in split_set and depth == 0:
              segs.append(t[start:i]); start = i + 1
      segs.append(t[start:])
      return segs

    for fid, raw in enumerate(df[text_col].fillna("").astype(str)):
        items = []
        def rec(t, d):
            t = t.strip()
            if _wrapped(t) and d < max_depth:
                rec(t[1:-1], d + 1); return
            for seg in _segs(t):
                seg = NUM_PREFIX.sub("", seg).strip().strip(JUNK_STRIP)
                if not seg:
                    continue
                if _wrapped(seg) and d < max_depth:
                    rec(seg[1:-1], d + 1); continue
                if seg in ITEM_STOP or DISCLAIMER_KW.search(seg):
                    continue
                items.extend(apply_mask(seg))
        rec(raw, 0)
        for pos, seg in enumerate(items, 1):
            rows.append((fid, seg, pos))

    out = pd.DataFrame(rows, columns=["fid", "item", "pos"])
    out["pos_bin"] = out["pos"].where(out["pos"] <= 10, 11)   # 11 = outside the first ten
    return out

# ===== Pipeline =====
core = pd.read_csv(INPUT_DIR + INPUT_FILE)
# add a step that screens whether an item is core
core = core[core['still_in_core']==1]

items_core = explode_items(core, "post_biz_main_clean")
print(f"firms: {len(core)}, items: {len(items_core)}")



# ---- 1. Frequency and position distribution of all items ----
ct = pd.crosstab(items_core["item"], items_core["pos_bin"])\
       .reindex(columns=range(1, 12), fill_value=0)
ct.columns = [f"pos{i}" for i in range(1, 11)] + ["pos_gt10"]

core_all = ct.copy()
core_all.insert(0, "freq", ct.sum(axis=1))
pos_stats = items_core.groupby("item")["pos"].agg(
    mean_pos="mean", min_pos="min", max_pos="max", median_pos="median")
pos_stats["mean_pos"] = pos_stats["mean_pos"].round(2)
core_all = core_all.join(pos_stats)
core_all = core_all.sort_values("freq", ascending=False)

core_all.to_csv(OUTPUT_DIR + OUTPUT_ALLITEMS, encoding="utf-8-sig")
print(f"{OUTPUT_ALLITEMS}: {len(core_all)} distinct phrases")

# ---- 2. Item frequency ranking at each position (1 to 10, and beyond) ----
blocks = []
for pb in range(1, 12):
    vc = items_core.loc[items_core["pos_bin"] == pb, "item"].value_counts()
    blk = vc.rename("freq").reset_index().rename(columns={"index": "item"})
    blk.insert(0, "position", f"pos{pb}" if pb <= 10 else "pos_gt10")
    blk.insert(1, "rank", range(1, len(blk) + 1))
    blocks.append(blk)

core_bypos_all = pd.concat(blocks, ignore_index=True)
core_bypos_all.to_csv(OUTPUT_DIR + OUTPUT_BYPOS,
                      index=False, encoding="utf-8-sig")
print(f"{OUTPUT_BYPOS}: {len(core_bypos_all)} rows")

## Unmasked version

In [ ]:
import re
import pandas as pd
INPUT_DIR = CORE_DIR
OUTPUT_DIR = INTERMEDIATE_DIR + "Results/"
INPUT_FILE = '0.03_test_4.18 Iterative Similarity(2).csv'
OUTPUT_ALLITEMS = 'beforemask_allitems_with_pos.csv'  # when rerunning, confirm the filter first
OUTPUT_BYPOS = 'beforemask_allitems_by_position.csv'

MASK_FILE_SPECIAL = pd.read_excel(HUMANLY_PICKED_DIR + '1. item split results.xlsx')
MASK_FILE_NORMAL  = pd.read_stata(HUMANLY_PICKED_DIR + '1. 按顿号split.dta')
# split by the hand-made special map
SPECIAL_MAP = (MASK_FILE_SPECIAL
               .dropna(subset=['item', 'split'])
               .astype({'item': str, 'split': str})
               .groupby('item', sort=False)['split']
               .apply(list)
               .to_dict())
# split on enumeration commas (、): NORMAL_SET
NORMAL_SET = set(MASK_FILE_NORMAL['item'].dropna().astype(str))

print(f"special covers {len(SPECIAL_MAP)}, normal split on the enumeration comma: {len(NORMAL_SET)}")

def apply_mask(seg):
    return [seg]

# ===== Bracket-aware sentence splitting =====
OPENS  = set('(（[【')
CLOSES = set(')）]】')
NUM_PREFIX = re.compile(r"^[0-9一二三四五六七八九十]+[..、]")
ITEM_STOP = {
    "", "无", "的", "依法须经批准的项目", "经相关部门批准后方可开展经营活动",
    "不含许可经营项目", "国家有专项规定的除外",
}
JUNK_STRIP = ' ^〓*#"\'`~·-—_=|'
DISCLAIMER_KW = re.compile(
    r'依法须经批准|凭营业执照|登记备案后方可|批准后方可开展经营|批准后依批准的内容|'
    r'除许可业务外|市场主体自主选择经营|国务院决定规定|不得以任何方式公开募集|'
    r'无需行政审批|外商投资准入特别管理措施|应取得相关部门许可后方可经营')
MAIN_SPLIT = set(';；,，。.^〓')
FALLBACK_SPLIT = set()      #set('、')

def _wrapped(t):
    """Whether the whole segment is wrapped in one pair of brackets"""
    if not t or t[0] not in OPENS:
        return False
    depth = 0
    for i, ch in enumerate(t):
        if ch in OPENS:
            depth += 1
        elif ch in CLOSES:
            depth -= 1
            if depth == 0:
                return i == len(t) - 1
    return False

def explode_items(df, text_col, max_depth=3):
    """Bracket-aware splitting: no split at depth > 0; a segment wrapped in brackets is unwrapped and split again, recursively;
       boilerplate fragments are filtered by keyword at the item level. Output columns (fid, item, pos, pos_bin)."""
    rows = []
    def _segs(t):
      # decide which separator to split this segment on
      split_set = MAIN_SPLIT if any(ch in MAIN_SPLIT for ch in t) else FALLBACK_SPLIT
      segs, depth, start = [], 0, 0
      for i, ch in enumerate(t):
          if ch in OPENS:
              depth += 1
          elif ch in CLOSES:
              depth = max(0, depth - 1)
          elif ch in split_set and depth == 0:
              segs.append(t[start:i]); start = i + 1
      segs.append(t[start:])
      return segs

    for fid, raw in enumerate(df[text_col].fillna("").astype(str)):
        items = []
        def rec(t, d):
            t = t.strip()
            if _wrapped(t) and d < max_depth:
                rec(t[1:-1], d + 1); return
            for seg in _segs(t):
                seg = NUM_PREFIX.sub("", seg).strip().strip(JUNK_STRIP)
                if not seg:
                    continue
                if _wrapped(seg) and d < max_depth:
                    rec(seg[1:-1], d + 1); continue
                if seg in ITEM_STOP or DISCLAIMER_KW.search(seg):
                    continue
                items.extend(apply_mask(seg))
        rec(raw, 0)
        for pos, seg in enumerate(items, 1):
            rows.append((fid, seg, pos))

    out = pd.DataFrame(rows, columns=["fid", "item", "pos"])
    out["pos_bin"] = out["pos"].where(out["pos"] <= 10, 11)   # 11 = outside the first ten
    return out

# ===== Pipeline =====
core = pd.read_csv(INPUT_DIR + INPUT_FILE)
items_core = explode_items(core, "post_biz_main_clean")
print(f"firms: {len(core)}, items: {len(items_core)}")

# ---- 1. Frequency and position distribution of all items ----
ct = pd.crosstab(items_core["item"], items_core["pos_bin"])\
       .reindex(columns=range(1, 12), fill_value=0)
ct.columns = [f"pos{i}" for i in range(1, 11)] + ["pos_gt10"]

core_all = ct.copy()
core_all.insert(0, "freq", ct.sum(axis=1))
pos_stats = items_core.groupby("item")["pos"].agg(
    mean_pos="mean", min_pos="min", max_pos="max", median_pos="median")
pos_stats["mean_pos"] = pos_stats["mean_pos"].round(2)
core_all = core_all.join(pos_stats)
core_all = core_all.sort_values("freq", ascending=False)

core_all.to_csv(OUTPUT_DIR + OUTPUT_ALLITEMS, encoding="utf-8-sig")
print(f"{OUTPUT_ALLITEMS}: {len(core_all)} distinct phrases")

# ---- 2. Item frequency ranking at each position (1 to 10, and beyond) ----
blocks = []
for pb in range(1, 12):
    vc = items_core.loc[items_core["pos_bin"] == pb, "item"].value_counts()
    blk = vc.rename("freq").reset_index().rename(columns={"index": "item"})
    blk.insert(0, "position", f"pos{pb}" if pb <= 10 else "pos_gt10")
    blk.insert(1, "rank", range(1, len(blk) + 1))
    blocks.append(blk)

core_bypos_all = pd.concat(blocks, ignore_index=True)
core_bypos_all.to_csv(OUTPUT_DIR + OUTPUT_BYPOS,
                      index=False, encoding="utf-8-sig")
print(f"{OUTPUT_BYPOS}: {len(core_bypos_all)} rows")

# MASK create: file comparing items from core and non-core firms

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# ===== Paths and parameters =====
INPUT_DIR  = INTERMEDIATE_DIR + "Results/"
OUTPUT_DIR = INTERMEDIATE_DIR + "Results/"
CORE_INPUT     = 'in_core_aftermask_allitems_with_pos.csv'
NOT_CORE_INPUT = 'not_in_core_aftermask_allitems_with_pos.csv'
SIM_THRESHOLD  = 0.95
OUTPUT_PANEL   = f'mapping_panel_{SIM_THRESHOLD}.csv'

core     = pd.read_csv(INPUT_DIR + CORE_INPUT)
not_core = pd.read_csv(INPUT_DIR + NOT_CORE_INPUT)
core_items = core['item'].astype(str).tolist()
nc_items   = not_core['item'].astype(str).tolist()
print(f"core vocabulary: {len(core_items)}, not-core vocabulary: {len(nc_items)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-m3", device=device)

emb_core = model.encode(core_items, batch_size=64, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
emb_nc   = model.encode(nc_items, batch_size=64, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

sim = emb_nc @ emb_core.T

not_core['max_sim']    = sim.max(axis=1)
not_core['best_match'] = [core_items[i] for i in sim.argmax(axis=1)]
mask = sim >= SIM_THRESHOLD
core_arr = np.array(core_items)
not_core['matched'] = [core_arr[mask[i]].tolist() for i in range(len(not_core))]
not_core['n_matched'] = mask.sum(axis=1)

# record idx
col_idx = {name: j for j, name in enumerate(core_items)}

panel = not_core[['item', 'freq', 'max_sim', 'best_match', 'n_matched', 'matched']]\
        .explode('matched')\
        .rename(columns={'matched': 'matched_item'})

panel['pair_sim'] = [
    sim[i, col_idx[m]] if pd.notna(m) else np.nan
    for i, m in zip(panel.index, panel['matched_item'])
]
panel['is_best'] = (panel['matched_item'] == panel['best_match']).where(
    panel['matched_item'].notna())

panel = panel.reset_index(drop=True)
panel.to_csv(OUTPUT_DIR + OUTPUT_PANEL, index=False, encoding='utf-8-sig')
print(f"panel: {len(panel)} rows, of which {panel['matched_item'].notna().sum()} matched pairs, "
      f"{panel['matched_item'].isna().sum()} unmatched placeholders")
print(f"Output: {OUTPUT_PANEL}")

# CIS industry codes

## Word Count

In [ ]:
RAW_DIR = INTERMEDIATE_DIR + "Raw_Data/"
# INPUTS
df = pd.read_stata(RAW_DIR+'basic_info.dta')
df = df[['query_company_name','report_year','name','fare_scope']]
df1 = pd.read_csv(RAW_DIR+'云计算.csv')
df2 = pd.read_csv(RAW_DIR+'互联网.csv')
df3 = pd.read_csv(RAW_DIR+'人工智能.csv')
df4 = pd.read_csv(RAW_DIR+'信息技术.csv')
df5 = pd.read_csv(RAW_DIR+'大数据.csv')
df6 = pd.read_csv(RAW_DIR+'新医药.csv')
df7 = pd.read_csv(RAW_DIR+'新材料.csv')
df8 = pd.read_csv(RAW_DIR+'新能源.csv')
df9 = pd.read_csv(RAW_DIR+'生物.csv')
df10 = pd.read_csv(RAW_DIR+'高端装备.csv')
big_df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10], ignore_index=True)

# WORD LISTS
rigorous_word_list = big_df['加粗关键词'].dropna().unique().tolist()
rigorous_set = set(rigorous_word_list)
relaxed_set = set(['云计算','互联网','人工智能','信息技术','大数据','新医药','新材料','新能源','生物','高端装备'])

rigorous_word_list_new = big_df.loc[big_df['加粗关键词'].notna(), '标题'].dropna().unique().tolist()
rigorous_set_new = set(rigorous_word_list_new)

loose_word_list = big_df['标题'].dropna().unique().tolist()
loose_set = set(loose_word_list)

def find_word(text, word_list):
    if pd.isna(text):
        return 0
    words = re.split(r'[；;、，,。（）()]', text)
    words = [w.strip() for w in words if w.strip()]
    return sum(1 for w in words for kw in word_list if kw in w)

def remove_space(text):
    if pd.isna(text):
        return text
    return text.replace(' ', '')

df['fare_scope'] = df['fare_scope'].apply(remove_space)

df['rigorous_count'] = df['fare_scope'].apply(lambda x: find_word(x, rigorous_set))
print('first complete')
df['rigorous_count_new'] = df['fare_scope'].apply(lambda x: find_word(x, rigorous_set_new))
print('second complete')
df['relaxed_count'] = df['fare_scope'].apply(lambda x: find_word(x, relaxed_set))
print('third complete')
df['loose_count'] = df['fare_scope'].apply(lambda x: find_word(x, loose_set))
print('fourth complete')
df.to_csv(RAW_DIR + 'word_count.csv')


## Word Proportion

In [ ]:
RAW_DIR = INTERMEDIATE_DIR + "Raw_Data/"
RESULT_DIR = INTERMEDIATE_DIR + "Results/"

INPUT_FILE = RAW_DIR + 'word_count.csv'
OUTPUT_FILE = RAW_DIR + 'word_count_prop.csv'

df = pd.read_csv(INPUT_FILE, index_col=0)

print('Columns:', df.columns.tolist())
count_cols = ['rigorous_count', 'rigorous_count_new', 'relaxed_count','loose_count']

def count_total(text):
    if pd.isna(text) or str(text).strip() in ('', '-'):
        return 0
    words = re.split(r'[；;、，,。（）()]', text)
    words = [w.strip() for w in words if w.strip()]
    return len(words)

df['total_desc'] = df['fare_scope'].apply(count_total)

for col in count_cols:
    prop_col = col.replace('_count', '_prop')
    df[prop_col] = df[col] / df['total_desc']
    df[prop_col] = df[prop_col].fillna(0)

df.to_csv(OUTPUT_FILE)
print('saved to word_count_prop.csv')

## Trend plots: totals

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = RAW_DIR + 'word_count.csv'

df = pd.read_csv(INPUT_FILE)
df = df[df['report_year'] >= 2010]

count_cols = ['rigorous_count', 'relaxed_count', 'rigorous_count_new','loose_count']

# aggregate by year: for each count column, the total and the number of firms with count > 0
agg_dict = {}
for col in count_cols:
    agg_dict[col + '_total'] = (col, 'sum')
    agg_dict[col + '_have'] = (col, lambda x: (x > 0).sum())
agg_dict['n_companies'] = ('report_year', 'size')

yearly = df.groupby('report_year').agg(**agg_dict).reset_index()

# Figure 1: overall trend
plt.figure(figsize=(9, 5))
for col in count_cols:
    plt.plot(yearly['report_year'], yearly[col + '_total'], marker='o', label=col)
plt.title('Total count by year')
plt.xlabel('Year'); plt.ylabel('Total count')
plt.legend(); plt.show()

# Figure 2: divided by the number of firms
plt.figure(figsize=(9, 5))
for col in count_cols:
    avg_all = yearly[col + '_total'] / yearly['n_companies']
    plt.plot(yearly['report_year'], avg_all, marker='o', label=col)
plt.title('Average count per company (all companies)')
plt.xlabel('Year'); plt.ylabel('Avg count')
plt.legend(); plt.show()

# Figure 3: mean among firms with count > 0
plt.figure(figsize=(9, 5))
for col in count_cols:
    avg_have = yearly[col + '_total'] / yearly[col + '_have']
    plt.plot(yearly['report_year'], avg_have, marker='o', label=col)
plt.title('Average count among companies with count > 0')
plt.xlabel('Year'); plt.ylabel('Avg count')
plt.legend(); plt.show()

## Trend plots: backfilled totals

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = RAW_DIR + 'word_count.csv'

df = pd.read_csv(INPUT_FILE)

count_cols = ['rigorous_count', 'relaxed_count', 'rigorous_count_new','loose_count']
id_col = 'query_company_name'   # <- replace with your firm identifier column

# treat 0 as missing; within each firm, sorted by year, backfill (later years fill earlier ones)
df = df.sort_values([id_col, 'report_year'])
for col in count_cols:
    s = df[col].replace(0, pd.NA)                      # 0 treated as missing
    df[col] = s.groupby(df[id_col]).bfill().fillna(0)  # backfill within each firm; values still empty go back to 0

# keep 2010 onward (after filling, since filling uses all later years)
df = df[df['report_year'] >= 2010]

# aggregate by year
agg_dict = {}
for col in count_cols:
    agg_dict[col + '_total'] = (col, 'sum')
    agg_dict[col + '_have'] = (col, lambda x: (x > 0).sum())
agg_dict['n_companies'] = ('report_year', 'size')

yearly = df.groupby('report_year').agg(**agg_dict).reset_index()

# Figure 1: overall trend
plt.figure(figsize=(9, 5))
for col in count_cols:
    plt.plot(yearly['report_year'], yearly[col + '_total'], marker='o', label=col)
plt.title('Total count by year (backfilled)')
plt.xlabel('Year'); plt.ylabel('Total count')
plt.legend(); plt.show()

# Figure 2: divided by the number of firms
plt.figure(figsize=(9, 5))
for col in count_cols:
    avg_all = yearly[col + '_total'] / yearly['n_companies']
    plt.plot(yearly['report_year'], avg_all, marker='o', label=col)
plt.title('Average count per company (all companies)')
plt.xlabel('Year'); plt.ylabel('Avg count')
plt.legend(); plt.show()

# Figure 3: mean among firms with count > 0
plt.figure(figsize=(9, 5))
for col in count_cols:
    avg_have = yearly[col + '_total'] / yearly[col + '_have']
    plt.plot(yearly['report_year'], avg_have, marker='o', label=col)
plt.title('Average count among companies with count > 0')
plt.xlabel('Year'); plt.ylabel('Avg count')
plt.legend(); plt.show()

## Trend plots: balanced panel

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = RAW_DIR + 'word_count.csv'

df = pd.read_csv(INPUT_FILE)

count_cols = ['rigorous_count', 'relaxed_count', 'rigorous_count_new','loose_count']
id_col = 'query_company_name'   # <- replace with your firm identifier column

# keep firms with a row in every year 2015-2020 (balanced panel)
window = df[(df['report_year'] >= 2017) & (df['report_year'] <= 2022)]
years_per_company = window.groupby(id_col)['report_year'].nunique()
balanced_ids = years_per_company[years_per_company == 6].index   # all 6 years present

panel = window[window[id_col].isin(balanced_ids)]

print(f'Firms in the balanced panel: {len(balanced_ids)}')
print(panel['report_year'].value_counts().sort_index())   # should be the same count each year

# aggregate by year
agg_dict = {}
for col in count_cols:
    agg_dict[col + '_total'] = (col, 'sum')
    agg_dict[col + '_have'] = (col, lambda x: (x > 0).sum())
agg_dict['n_companies'] = ('report_year', 'size')

yearly = panel.groupby('report_year').agg(**agg_dict).reset_index()

# Figure 1: overall trend
plt.figure(figsize=(9, 5))
for col in count_cols:
    plt.plot(yearly['report_year'], yearly[col + '_total'], marker='o', label=col)
plt.title('Total count by year (balanced panel 2015-2020)')
plt.xlabel('Year'); plt.ylabel('Total count')
plt.legend(); plt.show()

# Figure 2: divided by the number of firms
plt.figure(figsize=(9, 5))
for col in count_cols:
    avg_all = yearly[col + '_total'] / yearly['n_companies']
    plt.plot(yearly['report_year'], avg_all, marker='o', label=col)
plt.title('Average count per company (balanced panel 2015-2020)')
plt.xlabel('Year'); plt.ylabel('Avg count')
plt.legend(); plt.show()

# Figure 3: mean among firms with count > 0
plt.figure(figsize=(9, 5))
for col in count_cols:
    avg_have = yearly[col + '_total'] / yearly[col + '_have']
    plt.plot(yearly['report_year'], avg_have, marker='o', label=col)
plt.title('Average count among companies with count > 0 (balanced panel 2015-2020)')
plt.xlabel('Year'); plt.ylabel('Avg count')
plt.legend(); plt.show()

## Trend plots: proportions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = RAW_DIR + 'word_count_prop.csv'

df = pd.read_csv(INPUT_FILE, index_col=0)
df = df[df['report_year'] >= 2010]

# proportion columns (matching those created above)
prop_cols = ['rigorous_prop', 'rigorous_prop_new', 'relaxed_prop','loose_prop']

# aggregate by year: for each prop column, the mean share and the number of firms with share > 0
agg_dict = {}
for col in prop_cols:
    agg_dict[col + '_mean'] = (col, 'mean')                       # mean share over all firms
    agg_dict[col + '_have'] = (col, lambda x: (x > 0).sum())      # number of firms with share > 0
    agg_dict[col + '_sum']  = (col, 'sum')                        # sum of shares (for the mean among firms with share > 0)
agg_dict['n_companies'] = ('report_year', 'size')

yearly = df.groupby('report_year').agg(**agg_dict).reset_index()

# Figure 1: trend of the mean share (divided by the number of firms)
plt.figure(figsize=(9, 5))
for col in prop_cols:
    plt.plot(yearly['report_year'], yearly[col + '_mean'], marker='o', label=col)
plt.title('Average proportion per company (all companies)')
plt.xlabel('Year'); plt.ylabel('Avg proportion')
plt.legend(); plt.show()

# Figure 2: mean share among firms with share > 0
plt.figure(figsize=(9, 5))
for col in prop_cols:
    avg_have = yearly[col + '_sum'] / yearly[col + '_have']
    plt.plot(yearly['report_year'], avg_have, marker='o', label=col)
plt.title('Average proportion among companies with proportion > 0')
plt.xlabel('Year'); plt.ylabel('Avg proportion')
plt.legend(); plt.show()

# Step6 Split and Position

## Cell 0 core item initial position

This part of the code splits the business scope into different items and calculate the position statistics.

Meanwhile, it also defines the code needed in the following cells

In [ ]:
# =========================================================
# Cell 0 [baseline]: core item position list (own split) + shared functions
# =========================================================
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# ---------- Paths ----------
HUMANLY_PICKED_DIR = INTERMEDIATE_DIR + "Humanly_Picked_Data/"
INPUT_DIR  = CORE_DIR
OUTPUT_DIR = INTERMEDIATE_DIR + "Results/"

FILE_MAIN         = '0.03_test_4.18 Iterative Similarity(2).csv'
FILE_RAW          = 'benchmark.dta'
FILE_MASK_SPECIAL = '1. item split results 0810.xlsx'
FILE_MASK_NORMAL  = '1. 按顿号split 0810.dta'
FILE_MEANINGLESS = '1. 无意义0810(1).xlsx'

OUT_BASE = 'core_item_position_base.xlsx'

# ---------- Config ----------
MODEL_NAME = 'BAAI/bge-m3'
SIM_TH     = 0.85
MAX_K      = 20
FREQ_FLOOR = 1.0
TOP_N      = 4
VEC_SEP    = ' | '
SEP, J_KEEP, BATCH = '；', [1, 2, 3, 4], 64

OPENS  = set('(（[【')
CLOSES = set(')）]】')
NUM_PREFIX = re.compile(r"^[0-9一二三四五六七八九十]+[..、]")
ITEM_STOP = {"", "无", "的", "依法须经批准的项目",
             "经相关部门批准后方可开展经营活动", "不含许可经营项目",
             "国家有专项规定的除外"}
JUNK_STRIP = ' ^〓*#"\'`~·-—_=|'
DISCLAIMER_KW = re.compile(
    r'依法须经批准|凭营业执照|登记备案后方可|批准后方可开展经营|批准后依批准的内容|'
    r'除许可业务外|市场主体自主选择经营|国务院决定规定|不得以任何方式公开募集|'
    r'无需行政审批|外商投资准入特别管理措施|应取得相关部门许可后方可经营')
# boilerplate removed by regex: not an item, and takes no position
PRE_STRIP = re.compile(
    r'在?国家(?:法律)?(?:法规)?允许的?范围[内下]?(?:从事)?|'
    r'在?国家政策允许的?范围[内下]?')
MAIN_SPLIT     = set(';；,，。.^〓')
FALLBACK_SPLIT = set()

GENERAL = ["一般经营项目是：","一般经营项目是:","一般经营项目：","一般经营项目:",
           "一般经营项目 ：","一般经营项目 :","一般经营项目（","一般经营项目(",
           "一般项目：","一般项目:","一般项目 ：","一般项目 :",
           "一般事项：","一般事项:",
           "一般经营范围：","一般经营范围:","一般经营范围 ：","一般经营范围 :"]
PERMIT  = ["许可经营项目是：","许可经营项目是:","许可经营项目：","许可经营项目:",
           "许可经营项目 ：","许可经营项目 :","许可经营项目（","许可经营项目(",
           "许可项目：","许可项目:","许可项目 ：","许可项目 :",
           "许可经营范围：","许可经营范围:","许可经营范围 ：","许可经营范围 :"]
DISPLAY = ["自主展示（特色）项目：","自主展示（特色）项目:",
           "自主展示(特色)项目：","自主展示(特色)项目:",
           "自主展示项目：","自主展示项目:",
           "合伙协议记载的经营范围：","合伙协议记载的经营范围:",
           "协议记载的经营范围：","协议记载的经营范围:"]

# ---------- Mask: two layers + an optional third DP layer ----------
MASK_FILE_SPECIAL = pd.read_excel(HUMANLY_PICKED_DIR + FILE_MASK_SPECIAL)
MASK_FILE_NORMAL  = pd.read_stata(HUMANLY_PICKED_DIR + FILE_MASK_NORMAL)
_ml = pd.read_excel(HUMANLY_PICKED_DIR + FILE_MEANINGLESS)
MEANINGLESS = set(_ml['item'].dropna().astype(str).str.strip())
#-----
ITEM_STOP |= MEANINGLESS
print(f"{len(MEANINGLESS)} meaningless phrases added to ITEM_STOP, {len(ITEM_STOP)} in total")
#-----
SPECIAL_MAP = (MASK_FILE_SPECIAL.dropna(subset=['item', 'split'])
               .astype({'item': str, 'split': str})
               .groupby('item', sort=False)['split'].apply(list).to_dict())
NORMAL_SET = set(MASK_FILE_NORMAL['item'].dropna().astype(str))

USE_DP  = False        # True for undetermined only
SEGMENT = None         # Cell 3 injects the DP function

def apply_mask(seg):
    """Priority: manual special > manual split on the enumeration comma > (optional) DP item-list split > unchanged"""
    if seg in SPECIAL_MAP:
        return SPECIAL_MAP[seg]
    if seg in NORMAL_SET:
        return [p.strip() for p in seg.split('、') if p.strip()]
    if USE_DP and SEGMENT is not None:
        dp = SEGMENT(seg)
        if dp is not None:
            return dp
    return [seg]

print(f"mask: special {len(SPECIAL_MAP)}, normal {len(NORMAL_SET)}, DP={USE_DP}")

# ---------- Shared functions ----------
def save_xlsx(df, name, index=False):
    df.to_excel(OUTPUT_DIR + name, index=index)
    print(f"  -> {name} ({len(df)} rows)")

def first_pos(s, variants):
    for v in variants:
        p = s.find(v)
        if p >= 0:
            return p
    return -1

def decompose(scope):
    if pd.isna(scope):
        return []
    s = str(scope)
    hits = []
    for j, vs in [(1, GENERAL), (2, PERMIT), (4, DISPLAY)]:
        p = first_pos(s, vs)
        if p >= 0:
            hits.append((p, j))
    if not hits:
        return [(3, s.strip())]
    hits.sort()
    segs = []
    if hits[0][0] > 0:
        segs.append((3, s[:hits[0][0]].strip()))
    for k, (pos, j) in enumerate(hits):
        end = hits[k+1][0] if k+1 < len(hits) else len(s)
        segs.append((j, s[pos:end].strip()))
    return [(j, t) for j, t in segs if t != '']

def build_order(raw_df, id_col='id', scope_col='scope'):
    rows = []
    for i, s in zip(raw_df[id_col].values, raw_df[scope_col].values):
        for seq, (j, seg) in enumerate(decompose(s), 1):
            rows.append((i, j, seq))
    return pd.DataFrame(rows, columns=[id_col, 'j', 'seq'])

def _wrapped(t):
    if not t or t[0] not in OPENS:
        return False
    depth = 0
    for i, ch in enumerate(t):
        if ch in OPENS:
            depth += 1
        elif ch in CLOSES:
            depth -= 1
            if depth == 0:
                return i == len(t) - 1
    return False

def explode_items(df, text_col, max_depth=3):
    """Bracket-aware splitting + PRE_STRIP boilerplate removal + mask. Output (fid, item, pos)."""
    rows = []
    def _segs(t):
        split_set = MAIN_SPLIT if any(ch in MAIN_SPLIT for ch in t) else FALLBACK_SPLIT
        segs, depth, start = [], 0, 0
        for i, ch in enumerate(t):
            if ch in OPENS:
                depth += 1
            elif ch in CLOSES:
                depth = max(0, depth - 1)
            elif ch in split_set and depth == 0:
                segs.append(t[start:i]); start = i + 1
        segs.append(t[start:])
        return segs
    for fid, raw_t in enumerate(df[text_col].fillna("").astype(str)):
        items = []
        def rec(t, d):
            t = t.strip()
            if _wrapped(t) and d < max_depth:
                rec(t[1:-1], d + 1); return
            for seg in _segs(t):
                seg = NUM_PREFIX.sub("", seg)
                seg = PRE_STRIP.sub("", seg).strip().strip(JUNK_STRIP)
                if not seg:
                    continue
                if _wrapped(seg) and d < max_depth:
                    rec(seg[1:-1], d + 1); continue
                if seg in ITEM_STOP or DISCLAIMER_KW.search(seg):
                    continue
                items.extend(x for x in (p.strip() for p in apply_mask(seg)) if x)
        rec(raw_t, 0)
        for pos, seg in enumerate(items, 1):
            rows.append((fid, seg, pos))
    return pd.DataFrame(rows, columns=["fid", "item", "pos"])

def merge_to_firm(seg_df, order_df, label, id_col='id',
                  text_col='post_biz_main_clean', extra=()):
    s = seg_df[seg_df['j'].isin(J_KEEP)]
    if order_df is not None:
        s = s.merge(order_df, on=[id_col, 'j'], how='left')
    if s['seq'].isna().any():
        print(f"  [{label}] warning: {s['seq'].isna().sum()} segments not matched to seq")
    s = s.sort_values([id_col, 'seq'], na_position='last')
    agg = {'scope_merged': (text_col,
                            lambda x: SEP.join(v for v in x.fillna('').astype(str) if v.strip())),
           'n_segments': (text_col, 'size')}
    for c in extra:
        if c in s.columns:
            agg[c] = (c, 'first')
    firm = s.groupby(id_col, sort=False).agg(**agg).reset_index()
    print(f"  [{label}] {len(s)} segments -> {len(firm)} firms")
    return firm

def item_stats(it):
    g = it.groupby('item')['pos']
    return pd.DataFrame({'freq': g.size(), 'mean_pos': g.mean().round(4),
                         'min_pos': g.min(), 'max_pos': g.max(),
                         'median_pos': g.median()}).sort_values('freq', ascending=False)

def weighted_map(items, label, include_self=True):
    """Map to the core vocabulary; weight = core frequency, weighted quantity = the core item's own mean_pos.
       Returns (mapped, unmatched)."""
    items = list(items)
    emb = MODEL.encode(items, batch_size=BATCH, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    S = emb @ CORE_EMB.T
    recs, unm = [], []
    for i, itm in enumerate(items):
        s = S[i]
        idx = np.where(s >= SIM_TH)[0]
        if not include_self and itm in CORE_IDX:
            idx = idx[idx != CORE_IDX[itm]]
        if len(idx) == 0:
            j = int(s.argmax())
            unm.append((itm, float(s[j]), CORE_ITEMS[j]))
            continue
        if len(idx) > MAX_K:
            idx = idx[np.argsort(-s[idx])[:MAX_K]]
        w  = np.maximum(CORE_FREQ[idx], FREQ_FLOOR)
        wp = float((w * CORE_POSV[idx]).sum() / w.sum())
        o  = np.argsort(-w)
        recs.append((itm, wp, len(idx), float(w.sum()),
                     float(s[idx].max()), float(w.max() / w.sum()),
                     VEC_SEP.join(CORE_ITEMS[idx[k]] for k in o),
                     VEC_SEP.join(f'{w[k]:.0f}' for k in o),
                     VEC_SEP.join(f'{s[idx[k]]:.3f}' for k in o)))
    mapped = pd.DataFrame(recs, columns=['item', 'wpos', 'n_cand', 'sum_wts',
                                         'max_sim', 'wshare',
                                         'cands', 'wts', 'sims'])
    unmatched = pd.DataFrame(unm, columns=['item', 'max_sim', 'nearest'])
    print(f"[{label}] {len(items)} item: mapped {len(mapped)} ({len(mapped)/len(items):.2%}), "
          f"unmatched {len(unmatched)}")
    if len(mapped):
        print(f"  mean candidates {mapped['n_cand'].mean():.2f}, "
              f"median share of the largest weight {mapped['wshare'].median():.2%}, "
              f"mean wpos {mapped['wpos'].mean():.3f}")
    return mapped, unmatched

def build_firm_vector(firm, it, pmap, label, id_col='id', extra=()):
    """Firm-level top-4 position vector: one column per position plus one summary column."""
    it = it.copy()
    it['wpos'] = it['item'].map(pmap)
    top = it[it['pos'] <= TOP_N]

    cols = [id_col] + [c for c in extra if c in firm.columns] + ['scope_merged', 'n_segments']
    out = firm[cols].rename(columns={'scope_merged': '原表述'}).copy()
    out.index = range(len(firm)); out.index.name = 'fid'

    for src, pre in [('item', 'item'), ('wpos', 'pos')]:
        w = top.pivot(index='fid', columns='pos', values=src)\
               .reindex(columns=range(1, TOP_N + 1))
        w.columns = [f'{pre}{i}' for i in range(1, TOP_N + 1)]
        out = out.join(w)

    icols = [f'item{i}' for i in range(1, TOP_N + 1)]
    pcols = [f'pos{i}'  for i in range(1, TOP_N + 1)]
    out['item_vector'] = out[icols].apply(
        lambda r: VEC_SEP.join('' if pd.isna(v) else str(v) for v in r), axis=1)
    out['pos_vector'] = out[pcols].apply(
        lambda r: VEC_SEP.join('' if pd.isna(v) else f'{v:.4f}' for v in r), axis=1)

    g = it.groupby('fid')
    out['n_items']     = g.size().reindex(out.index, fill_value=0)
    out['n_matched']   = g['wpos'].count().reindex(out.index, fill_value=0)
    out['n_unmatched'] = out['n_items'] - out['n_matched']
    out['full_match']  = (out['n_unmatched'] == 0).astype(int)
    out['top4_mean']   = out[pcols].mean(axis=1, skipna=True).round(4)
    out['total_mean']  = g['wpos'].mean().round(4).reindex(out.index)
    out['sample']      = label

    n_full = (out[pcols].notna().sum(axis=1) == TOP_N).sum()
    print(f"  [{label}] firms {len(out)}, all four positions {n_full} ({n_full/len(out):.2%}), "
          f"full_match {out['full_match'].mean():.2%}")
    return out

# ---------- Core: firm-level split ----------
main = pd.read_csv(INPUT_DIR + FILE_MAIN)
raw  = pd.read_stata(RAW_DIR + FILE_RAW, columns=['id', 'scope'])
ORDER = build_order(raw)
print(f"Order restored for {len(ORDER)} segments")

flag = main.groupby('id')['still_in_core'].agg(['sum', 'size'])
flag.columns = ['n_core_seg', 'n_seg']
CORE_IDS = flag[flag['n_core_seg'] == flag['n_seg']].index
PEEL_IDS = flag[flag['n_core_seg'] <  flag['n_seg']].index
print(f"firms {len(flag)}: core {len(CORE_IDS)}, not-in-core {len(PEEL_IDS)} "
      f"({len(PEEL_IDS)/len(flag):.2%})")

# ---------- Core: baseline position list ----------
USE_DP = False
EXTRA_BM = ('原文件导入名称', '统一社会信用代码', 'indicator')
firm_core = merge_to_firm(main[main['id'].isin(CORE_IDS)], ORDER, 'core', extra=EXTRA_BM)
items_core = explode_items(firm_core, 'scope_merged')
base = item_stats(items_core)
save_xlsx(base, OUT_BASE, index=True)
print(f"core base: {len(items_core)} items, {len(base)} unique")

# ---------- Global constants: weighting base shared by the three branches ----------
CORE_ITEMS = base.index.astype(str).tolist()
CORE_IDX   = {t: i for i, t in enumerate(CORE_ITEMS)}
CORE_POSV  = base['mean_pos'].values.astype(np.float32)     # position from its own split
CORE_FREQ  = base['freq'].values.astype(np.float32)          # number of occurrences
print(f"core vocabulary {len(CORE_ITEMS)}, median freq {np.median(CORE_FREQ):.0f}, "
      f"max {CORE_FREQ.max():.0f}, mean pos {CORE_POSV.mean():.3f}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL  = SentenceTransformer(MODEL_NAME, device=DEVICE)
CORE_EMB = MODEL.encode(CORE_ITEMS, batch_size=BATCH, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
print(f"core embedding {CORE_EMB.shape}, device {DEVICE}")
print(f"Not caught: {MEANINGLESS & set(items_core['item'])}")

## Cell 1 core self map

In [ ]:
# ===== Cell 1: core 0.85 self-mapping (including itself) + firm-level top-4 vector =====
USE_DP = False
OUT_MAP  = 'core_item_position_mapped.xlsx'
OUT_UNM  = 'core_unmatched.xlsx'
OUT_VEC  = 'core_firm_position_vector.xlsx'
INCLUDE_SELF = True

mapped, unmatched = weighted_map(CORE_ITEMS, 'core', include_self=INCLUDE_SELF)
mapped = mapped.merge(base[['freq', 'mean_pos']].rename(columns={'mean_pos': 'pos_self'}),
                      left_on='item', right_index=True, how='left')
mapped['delta'] = mapped['wpos'] - mapped['pos_self']

PMAP_CORE = mapped.set_index('item')['wpos']
vec_core = build_firm_vector(firm_core, items_core, PMAP_CORE, 'core', extra=EXTRA_BM)

save_xlsx(mapped, OUT_MAP)
save_xlsx(unmatched, OUT_UNM)
save_xlsx(vec_core, OUT_VEC, index=True)
print(f"self-mapped vs own: corr {mapped['wpos'].corr(mapped['pos_self']):.4f}, "
      f"mean |delta| {mapped['delta'].abs().mean():.4f}, "
      f"median {mapped['delta'].abs().median():.4f}")

## Cell 2 peel out map

In [ ]:
# ===== Cell 2: not-in-core 0.85 mapping + firm-level top-4 vector =====
USE_DP = False
OUT_MAP = 'notincore_item_position_mapped.xlsx'
OUT_UNM = 'notincore_unmatched.xlsx'
OUT_VEC = 'notincore_firm_position_vector.xlsx'

firm_nic = merge_to_firm(main[main['id'].isin(PEEL_IDS)], ORDER, 'not-in-core', extra=EXTRA_BM)
items_nic = explode_items(firm_nic, 'scope_merged')
stats_nic = item_stats(items_nic)
stats_nic['in_core_vocab'] = stats_nic.index.isin(CORE_IDX).astype(int)
print(f"not-in-core: {len(items_nic)} items, {len(stats_nic)} unique, "
      f"already in the core vocabulary {stats_nic['in_core_vocab'].sum()} "
      f"({stats_nic['in_core_vocab'].mean():.2%})")

mapped, unmatched = weighted_map(stats_nic.index.astype(str), 'not-in-core',
                                 include_self=True)
sfx = stats_nic[['freq', 'mean_pos', 'in_core_vocab']].rename(
    columns={'freq': 'freq_branch', 'mean_pos': 'pos_branch'})
mapped    = mapped.merge(sfx, left_on='item', right_index=True, how='left')
unmatched = unmatched.merge(sfx, left_on='item', right_index=True, how='left')\
                     .sort_values('freq_branch', ascending=False)

PMAP_NIC = mapped.set_index('item')['wpos']
vec_nic = build_firm_vector(firm_nic, items_nic, PMAP_NIC, 'not_in_core', extra=EXTRA_BM)

save_xlsx(mapped, OUT_MAP)
save_xlsx(unmatched, OUT_UNM)
save_xlsx(vec_nic, OUT_VEC, index=True)
print(f"unmatched items cover {unmatched['freq_branch'].sum()} occurrences "
      f"({unmatched['freq_branch'].sum()/len(items_nic):.2%} of item occurrences)")

## Cell 3 undetermined map

In [ ]:
# ===== Cell 3: undetermined -> core 0.85 mapping (with the third DP mask layer) + firm-level vector =====
FILE_UNDE = 'test_Bert_Undetermined_Post_fix.xlsx'
FILE_ITEM = '2. item list.dta'
OUT_MAP      = 'tocore_undetermined_item_position_mapped.xlsx'
OUT_UNM      = 'tocore_undetermined_unmatched.xlsx'
OUT_VEC      = 'tocore_undetermined_firm_position_vector.xlsx'
OUT_UD_ITEMS = 'undetermined_items_aftermask.xlsx'

# ---- DP split over combinations (cached), injected as the third layer of apply_mask ----
item_match = pd.read_stata(HUMANLY_PICKED_DIR + FILE_ITEM)
DP_SET  = set(item_match['item'].astype(str).str.strip())
MAX_TOK = max(s.count('、') for s in DP_SET) + 1
_seg_cache = {}

def segment(text, S=DP_SET, max_tok=MAX_TOK):
    if text in _seg_cache:
        return _seg_cache[text]
    toks = text.split('、')
    n = len(toks)
    res = None
    if n > 1:
        best = [None] * (n + 1)
        best[0] = (0, -1, None)
        for i in range(1, n + 1):
            for j in range(max(0, i - max_tok), i):
                if best[j] is None:
                    continue
                seg = '、'.join(toks[j:i])
                if seg in S:
                    cand = (best[j][0] + 1, j, seg)
                    if best[i] is None or cand[0] < best[i][0]:
                        best[i] = cand
        if best[n] is not None:
            out, i = [], n
            while i > 0:
                _, j, sgm = best[i]
                out.append(sgm); i = j
            if len(out) > 1:
                res = out[::-1]
    _seg_cache[text] = res
    return res

SEGMENT, USE_DP = segment, True
print(f"DP vocabulary {len(DP_SET)}, MAX_TOK {MAX_TOK}, USE_DP={USE_DP}")

# ---- Read in and merge at the firm level ----
unde = pd.read_excel(UNDETERMINED_DIR + FILE_UNDE)
EXTRA_UD = ('stock_name', '系统匹配企业名称', '统一社会信用代码', '工商注册号')
firm_u = merge_to_firm(unde, None, 'undetermined', id_col='id', extra=EXTRA_UD)
items_u = explode_items(firm_u, 'scope_merged')
stats_u = item_stats(items_u)
stats_u['in_core_vocab'] = stats_u.index.isin(CORE_IDX).astype(int)
save_xlsx(stats_u, OUT_UD_ITEMS, index=True)
print(f"undetermined: {len(items_u)} items, {len(stats_u)} unique, "
      f"DP split applied to {sum(1 for v in _seg_cache.values() if v)} combinations, "
      f"already in the core vocabulary {stats_u['in_core_vocab'].sum()} ({stats_u['in_core_vocab'].mean():.2%})")

# ---- Mapping ----
mapped, unmatched = weighted_map(stats_u.index.astype(str), 'undetermined',
                                 include_self=True)
sfx = stats_u[['freq', 'mean_pos', 'in_core_vocab']].rename(
    columns={'freq': 'freq_branch', 'mean_pos': 'pos_branch'})
mapped    = mapped.merge(sfx, left_on='item', right_index=True, how='left')
unmatched = unmatched.merge(sfx, left_on='item', right_index=True, how='left')\
                     .sort_values('freq_branch', ascending=False)

PMAP_UD = mapped.set_index('item')['wpos']
vec_ud = build_firm_vector(firm_u, items_u, PMAP_UD, 'undetermined', extra=EXTRA_UD)

save_xlsx(mapped, OUT_MAP)
save_xlsx(unmatched, OUT_UNM)
save_xlsx(vec_ud, OUT_VEC, index=True)
print(f"unmatched {len(unmatched)}, covering {unmatched['freq_branch'].sum()} occurrences "
      f"({unmatched['freq_branch'].sum()/len(items_u):.2%} of item occurrences)")
print(f"of which {(unmatched['max_sim'] < .7).sum()} have max_sim<0.7 (no true counterpart)")

USE_DP = False          # switch off after use

## Cell 4 split after mask

In [ ]:
# ===== Cell 4: item list after the mask split (full benchmark + undetermined) =====
OUT_BM, OUT_UD = 'benchmark_items_aftermask.csv', 'dpcore_undetermined_items_aftermask.csv'

USE_DP = False                                    # benchmark: two layers
firm_bm  = merge_to_firm(main, ORDER, 'benchmark (all)')
items_bm = explode_items(firm_bm, 'scope_merged')
stats_bm = item_stats(items_bm)
stats_bm['in_core_vocab'] = stats_bm.index.isin(CORE_IDX).astype(int)
stats_bm.to_csv(OUTPUT_DIR + OUT_BM, encoding='utf-8-sig')
print(f"benchmark: {len(firm_bm)} firms, {len(items_bm)} items, {len(stats_bm)} unique -> {OUT_BM}")

USE_DP = True                                     # undetermined: three layers
stats_u2 = item_stats(explode_items(firm_u, 'scope_merged'))
stats_u2['in_core_vocab'] = stats_u2.index.isin(CORE_IDX).astype(int)
stats_u2.to_csv(OUTPUT_DIR + OUT_UD, encoding='utf-8-sig')
print(f"undetermined: {len(stats_u2)} unique -> {OUT_UD}")

USE_DP = False

## Step5 one time

In [ ]:
# =========================================================
# Step 5 [auxiliary]: full item lists for benchmark / undetermined
# =========================================================
OUT_BM_ITEMS = 'aux_benchmark_items_aftermask.xlsx'
OUT_UD_ITEMS = 'aux_undetermined_items_afterdp.xlsx'

PUNCT = re.compile(r'[、,，;；。..:：()（）\[\]【】/\\|\s]')

def add_no_punct(stats):
    stats = stats.copy()
    stats['no_punct'] = (~stats.index.astype(str).str.contains(PUNCT)).astype(int)
    return stats

def max_sim_to_core(items, label):
    emb = MODEL.encode(list(items), batch_size=BATCH, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    S = emb @ CORE_EMB.T
    j = S.argmax(axis=1)
    print(f"  [{label}] median max_sim {np.median(S.max(axis=1)):.4f}, "
          f"share >=0.85: {(S.max(axis=1) >= SIM_TH).mean():.2%}")
    return S.max(axis=1), [CORE_ITEMS[k] for k in j]

# ---------- 1. Full benchmark (core + not-in-core, two mask layers) ----------
USE_DP = False
firm_bm  = merge_to_firm(main, ORDER, 'benchmark (all)', extra=EXTRA_BM)
firm_bm['subset'] = np.where(firm_bm['id'].isin(CORE_IDS), 'core', 'not_in_core')
items_bm = explode_items(firm_bm, 'scope_merged')
items_bm['subset'] = firm_bm['subset'].reindex(items_bm['fid']).values

stats_bm = add_no_punct(item_stats(items_bm))
sub_freq = (items_bm.pivot_table(index='item', columns='subset', values='pos',
                                 aggfunc='size', fill_value=0)
            .reindex(columns=['core', 'not_in_core'], fill_value=0)
            .rename(columns={'core': 'freq_core', 'not_in_core': 'freq_nic'}))
stats_bm = stats_bm.join(sub_freq)
stats_bm['nic_share']     = (stats_bm['freq_nic'] / stats_bm['freq']).round(4)
stats_bm['in_core_vocab'] = stats_bm.index.isin(CORE_IDX).astype(int)

save_xlsx(stats_bm, OUT_BM_ITEMS, index=True)
print(f"benchmark: {len(firm_bm)} firms, {len(items_bm)} items, {len(stats_bm)} unique, "
      f"no punctuation {stats_bm['no_punct'].sum()} ({stats_bm['no_punct'].mean():.2%}), "
      f"from not-in-core only {(stats_bm['freq_core'] == 0).sum()}")

# ---------- 2. Full undetermined (DP + two mask layers) ----------
assert SEGMENT is not None, "Run Cell 3 first to define SEGMENT"
USE_DP = True
firm_u2  = merge_to_firm(unde, ORDER_U, 'undetermined', id_col='id', extra=EXTRA_UD)
items_u2 = explode_items(firm_u2, 'scope_merged')

stats_u2 = add_no_punct(item_stats(items_u2))
stats_u2['in_core_vocab'] = stats_u2.index.isin(CORE_IDX).astype(int)
stats_u2['max_sim'], stats_u2['nearest_core'] = max_sim_to_core(
    stats_u2.index.astype(str), 'undetermined')
stats_u2 = stats_u2.rename(columns={'mean_pos': 'pos_unde'})

save_xlsx(stats_u2, OUT_UD_ITEMS, index=True)
print(f"undetermined: {len(firm_u2)} firms, {len(items_u2)} items, {len(stats_u2)} unique, "
      f"no punctuation {stats_u2['no_punct'].sum()} ({stats_u2['no_punct'].mean():.2%}), "
      f"already in the core vocabulary {stats_u2['in_core_vocab'].sum()} ({stats_u2['in_core_vocab'].mean():.2%})")

USE_DP = False

## Step6 unde all

In [ ]:
# =========================================================
# Cell 6: raw-row level (207,868): identifiers + business scope + all top-4 mapped attributes
# =========================================================
FILE_HP   = '合伙部分.dta'
OUT_ROW   = 'tocore_undetermined_rowlevel_position.xlsx'
DEDUP_KEY = ['系统匹配企业名称', '统一社会信用代码', '经营范围']
ID_COLS   = ['stock_name', '系统匹配企业名称', '统一社会信用代码', '工商注册号']
# item-level attributes carried at each position (wpos is already pos{i}, not repeated)
ATTR_COLS = ['n_cand', 'sum_wts', 'wshare', 'max_sim',
             'freq_branch', 'pos_branch', 'in_core_vocab', 'cands', 'wts', 'sims']
TO_CSV    = False        # True writes csv instead: much faster, with no loss of precision

# ---- 1. Raw rows + restored id ----
hp = pd.read_stata(RAW_DIR + FILE_HP)
for c in ID_COLS:
    hp[c] = hp[c].astype(str).str.strip()
n_raw = len(hp)
uniq = hp.drop_duplicates(DEDUP_KEY).reset_index(drop=True)
uniq['id'] = 'P' + (uniq.index + 1).astype(str)
row = hp.merge(uniq[DEDUP_KEY + ['id']], on=DEDUP_KEY, how='left')
assert len(row) == n_raw and row['id'].notna().all(), "error when merging the id back"
print(f"raw rows {n_raw}, unique firms {len(uniq)}")

# ---- 2. Firm-level vectors ----
if 'vec_ud' in dir():
    v = vec_ud.reset_index(drop=True).copy()
else:
    v = pd.read_excel(OUTPUT_DIR + 'tocore_undetermined_firm_position_vector.xlsx',
                      dtype={'id': str})
POS_COLS  = [f'pos{i}'  for i in range(1, TOP_N + 1)]
ITEM_COLS = [f'item{i}' for i in range(1, TOP_N + 1)]
v = v[['id'] + ITEM_COLS + POS_COLS + ['原表述', 'pos_vector', 'n_items',
                                       'n_matched', 'n_unmatched', 'full_match',
                                       'top4_mean', 'total_mean']]

# ---- 3. Item-level mapped attributes ----
if 'mapped' in dir() and 'freq_branch' in mapped.columns:
    mp = mapped.copy()
else:
    mp = pd.read_excel(OUTPUT_DIR + 'tocore_undetermined_item_position_mapped.xlsx')
ATTR = mp.drop_duplicates('item').set_index('item')[ATTR_COLS]
print(f"mapping table: {len(ATTR)} items, {len(ATTR_COLS)} attribute columns")

# ---- 4. Combine ----
out = row.merge(v, on='id', how='left')
for i in range(1, TOP_N + 1):
    out = out.join(ATTR.add_suffix(f'_{i}'), on=f'item{i}')
assert len(out) == n_raw, f"row count changed: {len(out)} != {n_raw}"

per_pos = []
for i in range(1, TOP_N + 1):
    per_pos += [f'item{i}'] + [f'{c}_{i}' for c in ATTR_COLS]
out = out[['id'] + ID_COLS + ['经营范围', '原表述'] + POS_COLS
          + ['pos_vector', 'n_items', 'n_matched', 'n_unmatched', 'full_match',
             'top4_mean', 'total_mean'] + per_pos]
out = out.rename(columns={'经营范围': '经营范围_原始', '原表述': '经营范围_清洗后'})

print(f"output {len(out)} rows x {len(out.columns)} columns, "
      f"pos1 missing {out['pos1'].isna().sum()} ({out['pos1'].isna().mean():.2%}), "
      f"full_match {out['full_match'].mean():.2%}")
if TO_CSV:
    name = OUT_ROW.replace('.xlsx', '.csv')
    out.to_csv(OUTPUT_DIR + name, index=False, encoding='utf-8-sig')
    print(f"  -> {name}")
else:
    save_xlsx(out, OUT_ROW)
del hp, uniq, row

## Step add count for paper

In [ ]:
# =========================================================
# Count: occurrences of 创业投资 (venture capital) in core / benchmark
# =========================================================
TERM = '创业投资'

# ---- core: exact item match ----
hit = items_core[items_core['item'] == TERM]
n_occ  = len(hit)
n_fund = hit['fid'].nunique()
print(f"[core] firms {len(firm_core)}")
print(f"  exact item '{TERM}': occurrence {n_occ}, funds {n_fund} "
      f"({n_fund/len(firm_core):.2%}), repeated in {n_occ - n_fund} firms")
print(f"  base table freq {base.loc[TERM, 'freq'] if TERM in base.index else 'NA'}, "
      f"mean_pos {base.loc[TERM, 'mean_pos'] if TERM in base.index else 'NA'}")
print(f"  position distribution: {hit['pos'].value_counts().sort_index().head(6).to_dict()}")

# ---- core: substring match (including 创业投资咨询, VC consulting) ----
sub = items_core[items_core['item'].str.contains(TERM, regex=False)]
print(f"  substring match: {sub['item'].nunique()} phrasings, occurrence {len(sub)}, "
      f"funds {sub['fid'].nunique()} ({sub['fid'].nunique()/len(firm_core):.2%})")
print(f"  most common: {sub['item'].value_counts().head(6).to_dict()}")

# ---- Overall duplicate rate ----
occ = items_core.groupby(['fid', 'item']).size()
print(f"  (firm, item) pairs {len(occ)}, repeated >1: {(occ>1).sum()} ({(occ>1).mean():.3%})")

# ---- Benchmark: full-sample denominator ----
print(f"\n[benchmark] firms {main['id'].nunique()}, "
      f"core {len(CORE_IDS)}, not-in-core {main['id'].nunique() - len(CORE_IDS)}")
bm = main.drop_duplicates('id')
print(f"  unique credit codes {bm['统一社会信用代码'].nunique()}, "
      f"unique firm names {bm['原文件导入名称'].nunique()}")